<a href="https://colab.research.google.com/github/jaimeisaac2020/Python-analsisis-basicos/blob/mi-github/Copia_de_dashboard_tesis_jaime_Mario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PASO 1: INSTALACIÓN Y CARGA DE LIBRERÍAS Y DATOS
# ==============================================================================
# !pip install pandas scikit-learn statsmodels ipywidgets matplotlib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
import ipywidgets as widgets
from ipywidgets import VBox, HBox, HTML, Output, link, Layout
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Cargar los datos
url = 'https://raw.githubusercontent.com/jaimeisaac2020/Python-analsisis-basicos/mi-github/dataset_completo_google_amazon.csv'
df = pd.read_csv(url, index_col='Date', parse_dates=True)
print("✅ Datos cargados correctamente.")

# ==============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS (PARA TODOS LOS ESCENARIOS)
# ==============================================================================
df_eng = df.copy()
# Características para Google y Amazon
for asset in ['GOOGL', 'AMZN']:
    df_eng[f'{asset}_Return_1d'] = df_eng[f'{asset}_Close'].pct_change()
    df_eng[f'{asset}_Volatility_20'] = df_eng[f'{asset}_Return_1d'].rolling(window=20).std()
    for lag in range(1, 6): df_eng[f'{asset}_Close_Lag_{lag}'] = df_eng[f'{asset}_Close'].shift(lag)
    df_eng[f'{asset}_SMA_5'] = df_eng[f'{asset}_Close'].rolling(window=5).mean()
    delta = df_eng[f'{asset}_Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df_eng[f'{asset}_RSI_14'] = 100 - (100 / (1 + rs))

df_final = df_eng.dropna()
print("✅ Ingeniería de características completada.")

# ==============================================================================
# PASO 3: ENTRENAMIENTO Y EVALUACIÓN DE TODOS LOS MODELOS DE LA TESIS
# ==============================================================================
results = {}
models = {}

# CORRECCIÓN: Definir la lista de features base ANTES del bucle
features_base = ['VIX', 'SP500', 'NASDAQ', 'Treasury_10Y', 'Inflacion_T5YIE']

for asset in ['GOOGL', 'AMZN']:
    results[asset] = {}
    models[asset] = {}

    # División de datos temporal
    train_size = int(len(df_final) * 0.8)
    train_df, test_df = df_final.iloc[:train_size], df_final.iloc[train_size:]

    y_train = train_df[f'{asset}_Close']
    y_test = test_df[f'{asset}_Close']

    # --- OBJETIVO 1 y 2: Tasas e Inflación ---
    # Modelo Base (solo VIX, SP500, NASDAQ)
    features_macro_base = ['VIX', 'SP500', 'NASDAQ']
    rf_base = RandomForestRegressor(random_state=42).fit(train_df[features_macro_base], y_train)
    preds = rf_base.predict(test_df[features_macro_base])
    results[asset]['RF_Base_Macro'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    # Modelo con Tasa
    features_con_tasa = features_macro_base + ['Treasury_10Y']
    rf_completo = RandomForestRegressor(random_state=42).fit(train_df[features_con_tasa], y_train)
    preds = rf_completo.predict(test_df[features_con_tasa])
    results[asset]['RF_Con_Tasa'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    # Modelo sin Inflación
    results[asset]['RF_Sin_Inflacion'] = results[asset]['RF_Con_Tasa'] # Es el mismo modelo

    # Modelo con Inflación
    features_con_inflacion = features_con_tasa + ['Inflacion_T5YIE']
    rf_con_inf = RandomForestRegressor(random_state=42).fit(train_df[features_con_inflacion], y_train)
    preds = rf_con_inf.predict(test_df[features_con_inflacion])
    results[asset]['RF_Con_Inflacion'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    # --- OBJETIVO 3 y 4: Históricos y Comparación Final ---
    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base]
    features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]

    X_train_enriched, y_train_enriched = train_df[features_enriched], train_df[f'{asset}_Close']
    X_test_enriched, y_test_enriched = test_df[features_enriched], test_df[f'{asset}_Close']

    # Modelos enriquecidos para la simulación
    models[asset]['Regresión Lineal'] = LinearRegression().fit(X_train_enriched, y_train_enriched)
    models[asset]['Random Forest'] = RandomForestRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Gradient Boosting'] = GradientBoostingRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)

    # Evaluación de modelos enriquecidos
    for name, model in models[asset].items():
        preds = model.predict(X_test_enriched)
        results[asset][f'{name}_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, preds), 'R2': r2_score(y_test_enriched, preds)}

    # Modelo ARIMA
    arima_order = (1, 1, 1) if asset == 'GOOGL' else (0, 1, 0)
    arima_model = ARIMA(y_train_enriched, order=arima_order).fit()
    arima_preds = arima_model.forecast(steps=len(y_test_enriched))
    results[asset]['ARIMA_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, arima_preds), 'R2': r2_score(y_test_enriched, arima_preds)}
    models[asset]['ARIMA_pred_estatica'] = arima_model.forecast(steps=1).iloc[0]

print("✅ Todos los modelos han sido entrenados y evaluados.")


# ==============================================================================
# PASO 4: CREACIÓN DEL DASHBOARD INTERACTIVO CON PESTAÑAS
# ==============================================================================
# (Esta sección es idéntica a la anterior, no necesita cambios)

# --- PESTAÑA 1: Impacto Tasas de Interés ---
tab1_output = Output()
def display_objective1(asset):
    with tab1_output:
        tab1_output.clear_output(wait=True)
        res_base = results[asset]['RF_Base_Macro']
        res_completo = results[asset]['RF_Con_Tasa']
        df_res = pd.DataFrame([res_base, res_completo], index=['Modelo Base', 'Modelo con Tasa Int.'])
        display(HTML(f"<h4>Impacto de Tasas de Interés en {asset}</h4>"))
        display(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='lightgreen'))

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(df_res.index, df_res['MSE'], color=['gray', 'skyblue'])
        ax.set_title('Comparación de Error (MSE)')
        plt.show()

# --- PESTAÑA 2: Impacto Inflación ---
tab2_output = Output()
def display_objective2(asset):
    with tab2_output:
        tab2_output.clear_output(wait=True)
        res_sin = results[asset]['RF_Sin_Inflacion']
        res_con = results[asset]['RF_Con_Inflacion']
        df_res = pd.DataFrame([res_sin, res_con], index=['Modelo SIN Inflación', 'Modelo CON Inflación'])
        incremento_error = (res_sin['MSE'] / res_con['MSE'] - 1) * 100
        display(HTML(f"<h4>Estudio de Ablación de Inflación en {asset}</h4>"))
        display(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='lightgreen'))
        display(HTML(f"<p>Eliminar la inflación <b>aumentó el error en un {incremento_error:.2f}%</b>.</p>"))

# --- PESTAÑA 3: Impacto Datos Históricos ---
tab3_output = Output()
def display_objective3(asset):
    with tab3_output:
        tab3_output.clear_output(wait=True)
        res_base = results[asset]['RF_Base_Macro']
        res_enriquecido = results[asset]['Random Forest_Enriquecido']
        df_res = pd.DataFrame([res_base, res_enriquecido], index=['Modelo SIN Históricos', 'Modelo CON Históricos'])
        display(HTML(f"<h4>Impacto de Datos Históricos en {asset}</h4>"))
        if asset == 'AMZN':
            display(HTML("<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>¡Resultado Clave!</b> El R² pasó de un valor muy negativo a uno positivo, transformando el modelo.</div>"))
        display(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='lightgreen').applymap(lambda x: 'color:red' if x<0 else 'color:green', subset=['R2']))

# --- PESTAÑA 4: Simulador Interactivo ---
asset_selector = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Seleccione el Activo:')
tasa_interes_slider = widgets.FloatSlider(value=4.0, min=2.0, max=6.0, step=0.05, description='Tasa de Interés:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
inflacion_slider = widgets.FloatSlider(value=2.5, min=1.5, max=4.0, step=0.05, description='Inflación:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
vix_slider = widgets.FloatSlider(value=15, min=10, max=40, step=1, description='VIX:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
precio_anterior_slider = widgets.FloatSlider(value=df_final['AMZN_Close'].iloc[-1], min=150, max=250, step=0.5, description='Precio Anterior:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
tasa_interes_text, inflacion_text, vix_text, precio_anterior_text = (widgets.FloatText(layout=Layout(width='20%')) for _ in range(4))
link((tasa_interes_slider, 'value'), (tasa_interes_text, 'value')); link((inflacion_slider, 'value'), (inflacion_text, 'value')); link((vix_slider, 'value'), (vix_text, 'value')); link((precio_anterior_slider, 'value'), (precio_anterior_text, 'value'))
table_output_container = HTML()
graph_output_container = Output()

def update_simulador(asset, tasa_interes, inflacion, vix, precio_anterior):
    if any(v is None for v in [tasa_interes, inflacion, vix, precio_anterior]): return
    last_known = df_final.iloc[-1].copy()
    input_data = pd.DataFrame([last_known])
    input_data['Treasury_10Y'], input_data['Inflacion_T5YIE'], input_data['VIX'], input_data[f'{asset}_Close_Lag_1'] = tasa_interes, inflacion, vix, precio_anterior
    for lag in range(2, 6): input_data[f'{asset}_Close_Lag_{lag}'] = precio_anterior
    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
    input_data_final = input_data[features_enriched]

    preds = {name: model.predict(input_data_final)[0] for name, model in models[asset].items() if name != 'ARIMA_pred_estatica'}
    preds['ARIMA'] = models[asset]['ARIMA_pred_estatica']

    table_output_container.value = create_html_table(preds, precio_anterior)

    graph_output_container.clear_output(wait=True)
    with graph_output_container:
        create_prediction_graph(df_final[f'{asset}_Close'], preds, asset, precio_anterior)

def create_html_table(preds, precio_anterior):
    best_model_name = "Random Forest"
    html_table = f"""<div style="border: 1px solid #ccc; padding: 15px; border-radius: 8px;"><h3 style="margin-top:0;">Tabla de Predicciones</h3><p style="font-size:0.9em;">Precio Anterior: <b>{precio_anterior:,.2f} USD</b></p><table style="width:100%; border-collapse: collapse;"><thead><tr style="background-color:#005a9c; color:white;"><th style="padding: 10px; text-align: left;">Modelo</th><th style="padding: 10px; text-align: right;">Precio Predicho</th></tr></thead><tbody>"""
    for model_name in ['Random Forest', 'Gradient Boosting', 'Regresión Lineal', 'ARIMA']:
        prediction = preds[model_name]
        is_best = (model_name == best_model_name)
        row_style = 'background-color:#d4edda; color:#155724; font-weight:bold;' if is_best else ''
        tag = ' <span style="font-size:0.8em; background-color:#28a745; color:white; padding: 2px 5px; border-radius:3px;">Mejor Modelo</span>' if is_best else ''
        static_tag = ' <span style="font-size:0.8em; background-color:#dc3545; color:white; padding: 2px 5px; border-radius:3px;">Estático</span>' if model_name == 'ARIMA' else ''
        html_table += f"""<tr style="{row_style}"><td style="padding: 8px; border-bottom:1px solid #ddd;">{model_name}{tag}{static_tag}</td><td style="padding: 8px; border-bottom:1px solid #ddd; text-align: right;">{prediction:,.2f}</td></tr>"""
    html_table += "</tbody></table></div>"
    return html_table

def create_prediction_graph(historic_data_series, preds, asset, precio_anterior):
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(10, 5))
    historic_data = historic_data_series.tail(100)
    ax.plot(historic_data.index, historic_data.values, label=f'Histórico de {asset}', color='black', alpha=0.7)
    last_date = historic_data.index[-1]
    prediction_date = last_date + pd.Timedelta(days=1)
    colors = {'ARIMA': 'red', 'Regresión Lineal': 'orange', 'Gradient Boosting': 'purple', 'Random Forest': 'blue'}
    best_model_name = "Random Forest"
    for model_name, prediction in preds.items():
        is_best = (model_name == best_model_name)
        ax.plot([last_date, prediction_date], [precio_anterior, prediction], linestyle='--', marker='o', color=colors[model_name], label=f'{model_name}: {prediction:,.2f}', linewidth=3.0 if is_best else 1.5, alpha=1.0 if is_best else 0.8)
    ax.set_title(f'Pronósticos para {asset} bajo Escenario Simulado', fontsize=16)
    ax.set_ylabel('Precio (USD)'); ax.legend(loc='best'); plt.tight_layout(); plt.show()

def on_asset_change_simulador(change):
    asset = change['new']
    last_price = df_final[f'{asset}_Close'].iloc[-1]
    precio_anterior_slider.min = last_price * 0.8; precio_anterior_slider.max = last_price * 1.2
    precio_anterior_slider.value = last_price

asset_selector.observe(on_asset_change_simulador, names='value')

# --- Construcción del UI del Dashboard Completo ---
interactive_ui = widgets.interactive_output(update_simulador, {'asset': asset_selector, 'tasa_interes': tasa_interes_slider, 'inflacion': inflacion_slider, 'vix': vix_slider, 'precio_anterior': precio_anterior_slider})
controles_sliders = VBox([ HBox([tasa_interes_slider, tasa_interes_text]), HBox([inflacion_slider, inflacion_text]), HBox([vix_slider, vix_text]), HBox([precio_anterior_slider, precio_anterior_text]) ])
controles_principales = VBox([asset_selector, HTML("<hr><p><b>Defina un escenario:</b></p>"), controles_sliders])
resultados_ui = HBox([table_output_container, graph_output_container], layout=Layout(align_items='flex-start', justify_content='space-between'))
tab4_content = VBox([controles_principales, resultados_ui])
on_asset_change_simulador({'new': asset_selector.value})

tabs = widgets.Tab()
tabs.children = [tab1_output, tab2_output, tab3_output, tab4_content]
tabs.set_title(0, '1. Impacto Tasas Interés'); tabs.set_title(1, '2. Impacto Inflación'); tabs.set_title(2, '3. Impacto Datos Históricos'); tabs.set_title(3, '4. DEMO INTERACTIVA')
asset_selector_static = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Activo para Análisis:')
widgets.interactive_output(display_objective1, {'asset': asset_selector_static})
widgets.interactive_output(display_objective2, {'asset': asset_selector_static})
widgets.interactive_output(display_objective3, {'asset': asset_selector_static})

display(HTML("<h1>Dashboard de Tesis: Análisis y Simulación</h1>"))
display(asset_selector_static)
display(tabs)

✅ Datos cargados correctamente.
✅ Ingeniería de características completada.
✅ Todos los modelos han sido entrenados y evaluados.


HTML(value='<h1>Dashboard de Tesis: Análisis y Simulación</h1>')

Dropdown(description='Activo para Análisis:', options=('AMZN', 'GOOGL'), value='AMZN')

In [2]:
# ==============================================================================
# PASO 1: INSTALACIÓN Y CARGA DE LIBRERÍAS Y DATOS
# ==============================================================================
# !pip install pandas scikit-learn statsmodels ipywidgets matplotlib
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
import ipywidgets as widgets
from ipywidgets import VBox, HBox, HTML, Output, link, Layout
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Cargar los datos
url = 'https://raw.githubusercontent.com/jaimeisaac2020/Python-analsisis-basicos/mi-github/dataset_completo_google_amazon.csv'
df = pd.read_csv(url, index_col='Date', parse_dates=True)
print("✅ Datos cargados correctamente.")

# ==============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS
# ==============================================================================
df_eng = df.copy()
for asset in ['GOOGL', 'AMZN']:
    df_eng[f'{asset}_Return_1d'] = df_eng[f'{asset}_Close'].pct_change()
    df_eng[f'{asset}_Volatility_20'] = df_eng[f'{asset}_Return_1d'].rolling(window=20).std()
    for lag in range(1, 6): df_eng[f'{asset}_Close_Lag_{lag}'] = df_eng[f'{asset}_Close'].shift(lag)

df_final = df_eng.dropna()
print("✅ Ingeniería de características completada.")

# ==============================================================================
# PASO 3: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS
# ==============================================================================
results = {}
models = {}
features_base_all = ['VIX', 'SP500', 'NASDAQ', 'Treasury_10Y', 'Inflacion_T5YIE']

for asset in ['GOOGL', 'AMZN']:
    results[asset] = {}
    models[asset] = {}

    train_size = int(len(df_final) * 0.8)
    train_df, test_df = df_final.iloc[:train_size], df_final.iloc[train_size:]
    y_train, y_test = train_df[f'{asset}_Close'], test_df[f'{asset}_Close']

    # --- Experimentos para Pestañas 1, 2, 3 ---
    features_macro_base = ['VIX', 'SP500', 'NASDAQ']
    rf_base = RandomForestRegressor(random_state=42).fit(train_df[features_macro_base], y_train); preds = rf_base.predict(test_df[features_macro_base])
    results[asset]['RF_Base_Macro'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_tasa = features_macro_base + ['Treasury_10Y']
    rf_con_tasa = RandomForestRegressor(random_state=42).fit(train_df[features_con_tasa], y_train); preds = rf_con_tasa.predict(test_df[features_con_tasa])
    results[asset]['RF_Con_Tasa'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_inflacion = features_con_tasa + ['Inflacion_T5YIE']
    rf_con_inf = RandomForestRegressor(random_state=42).fit(train_df[features_con_inflacion], y_train); preds = rf_con_inf.predict(test_df[features_con_inflacion])
    results[asset]['RF_Con_Inflacion'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    # --- Modelos Finales (Enriquecidos) para Pestaña 4 y Pestaña 3 ---
    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]
    features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]

    X_train_enriched, y_train_enriched = train_df[features_enriched], train_df[f'{asset}_Close']
    X_test_enriched, y_test_enriched = test_df[features_enriched], test_df[f'{asset}_Close']

    # Entrenar y guardar modelos
    models[asset]['Random Forest'] = RandomForestRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Gradient Boosting'] = GradientBoostingRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Regresión Lineal'] = LinearRegression().fit(X_train_enriched, y_train_enriched)

    # Evaluar y guardar resultados del modelo enriquecido (para Pestaña 3)
    preds_enriched = models[asset]['Random Forest'].predict(X_test_enriched)
    results[asset]['RF_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, preds_enriched), 'R2': r2_score(y_test_enriched, preds_enriched)}

    arima_order = (1, 1, 1) if asset == 'GOOGL' else (0, 1, 0)
    arima_model = ARIMA(y_train_enriched, order=arima_order).fit()
    models[asset]['ARIMA_pred_estatica'] = arima_model.forecast(steps=1).iloc[0]

print("✅ Todos los modelos han sido entrenados y evaluados.")


# ==============================================================================
# PASO 4: CREACIÓN DEL DASHBOARD INTERACTIVO CON PESTAÑAS
# ==============================================================================
def create_tab_objective1(asset):
    res_base = results[asset]['RF_Base_Macro']; res_completo = results[asset]['RF_Con_Tasa']
    df_res = pd.DataFrame([res_base, res_completo], index=['Modelo Base', 'Modelo con Tasa Int.'])
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    graph_out = Output()
    with graph_out:
        fig, ax = plt.subplots(figsize=(6, 4)); bars = ax.bar(df_res.index, df_res['MSE'], color=['gray', 'skyblue'])
        ax.set_title(f'Reducción de Error (MSE) al añadir Tasa de Interés para {asset}'); ax.bar_label(bars, fmt='{:,.2f}'); plt.show()
    return VBox([HTML("<h4>Validación H2: Impacto de Tasas de Interés</h4><p>Comparamos un modelo RF sin la tasa de interés vs. uno que la incluye.</p>"), styled_table_html, graph_out])

def create_tab_objective2(asset):
    res_sin = results[asset]['RF_Con_Tasa']; res_con = results[asset]['RF_Con_Inflacion']
    df_res = pd.DataFrame([res_sin, res_con], index=['Modelo SIN Inflación', 'Modelo CON Inflación'])
    incremento_error = (res_sin['MSE'] / res_con['MSE'] - 1) * 100 if res_con['MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    return VBox([HTML("<h4>Validación H3: Impacto de la Inflación (Estudio de Ablación)</h4><p>Comparamos un modelo con tasa de interés vs. uno que además incluye la inflación.</p>"),
                 styled_table_html,
                 HTML(f"<div style='margin-top:10px; padding:10px; background-color:#f8d7da; border-radius:5px;'>Eliminar la inflación <b>aumentó el error en un {incremento_error:.2f}%</b> para {asset}, demostrando su importancia crítica.</div>")])

def create_tab_objective3(asset):
    res_base = results[asset]['RF_Base_Macro']
    res_enriquecido = results[asset]['RF_Enriquecido'] # Usamos el resultado pre-calculado y guardado
    df_res = pd.DataFrame([res_base, res_enriquecido], index=['Modelo SIN Históricos', 'Modelo CON Históricos'])
    r2_color = 'color:red' if df_res.loc['Modelo CON Históricos', 'R2'] < 0 else 'color:green'
    html_resumen = (f"<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>¡Resultado Clave para {asset}!</b> El R² pasó de {res_base['R2']:.2f} a <b style='{r2_color}'>{res_enriquecido['R2']:.2f}</b>, transformando el modelo.</div>") if asset == 'AMZN' else ""
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())
    return VBox([HTML("<h4>Validación H4: Impacto de Datos Históricos</h4><p>Comparamos un modelo solo con variables macro vs. uno enriquecido.</p>"),
                 HTML(html_resumen),
                 styled_table_html])

# --- PESTAÑA 4: Simulador Interactivo ---
asset_selector_sim = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Seleccione el Activo:')
tasa_interes_slider = widgets.FloatSlider(value=4.0, min=2.0, max=6.0, step=0.05, description='Tasa Interés:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
inflacion_slider = widgets.FloatSlider(value=2.5, min=1.5, max=4.0, step=0.05, description='Inflación:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
vix_slider = widgets.FloatSlider(value=15, min=10, max=40, step=1, description='VIX:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
precio_anterior_slider = widgets.FloatSlider(value=df_final['AMZN_Close'].iloc[-1], min=150, max=250, step=0.5, description='Precio Anterior:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
tasa_interes_text, inflacion_text, vix_text, precio_anterior_text = (widgets.FloatText(layout=Layout(width='20%')) for _ in range(4))
link((tasa_interes_slider, 'value'), (tasa_interes_text, 'value')); link((inflacion_slider, 'value'), (inflacion_text, 'value')); link((vix_slider, 'value'), (vix_text, 'value')); link((precio_anterior_slider, 'value'), (precio_anterior_text, 'value'))
table_output_container = HTML(); graph_output_container = Output()

def update_simulador(asset, tasa_interes, inflacion, vix, precio_anterior):
    if any(v is None for v in [tasa_interes, inflacion, vix, precio_anterior]): return
    last_known = df_final.iloc[-1].copy(); input_data = pd.DataFrame([last_known])
    input_data['Treasury_10Y'], input_data['Inflacion_T5YIE'], input_data['VIX'], input_data[f'{asset}_Close_Lag_1'] = tasa_interes, inflacion, vix, precio_anterior
    for lag in range(2, 6): input_data[f'{asset}_Close_Lag_{lag}'] = precio_anterior
    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
    input_data_final = input_data[features_enriched]
    preds = {name: model.predict(input_data_final)[0] for name, model in models[asset].items() if name != 'ARIMA_pred_estatica'}; preds['ARIMA'] = models[asset]['ARIMA_pred_estatica']
    table_output_container.value = create_html_table(preds, precio_anterior)
    graph_output_container.clear_output(wait=True)
    with graph_output_container: create_prediction_graph(df_final[f'{asset}_Close'], preds, asset, precio_anterior)

def create_html_table(preds, precio_anterior):
    best_model_name = "Random Forest"
    html_table = f"""<div style="border: 1px solid #ccc; padding: 15px; border-radius: 8px;"><h3 style="margin-top:0;">Tabla de Predicciones</h3><p style="font-size:0.9em;">Precio Anterior: <b>{precio_anterior:,.2f} USD</b></p><table style="width:100%; border-collapse: collapse;"><thead><tr style="background-color:#005a9c; color:white;"><th style="padding: 10px; text-align: left;">Modelo</th><th style="padding: 10px; text-align: right;">Precio Predicho</th></tr></thead><tbody>"""
    for model_name in ['Random Forest', 'Gradient Boosting', 'Regresión Lineal', 'ARIMA']:
        prediction = preds[model_name]
        is_best = (model_name == best_model_name); row_style = 'background-color:#d4edda; color:#155724; font-weight:bold;' if is_best else ''
        tag = ' <span style="font-size:0.8em; background-color:#28a745; color:white; padding: 2px 5px; border-radius:3px;">Mejor Modelo</span>' if is_best else ''
        static_tag = ' <span style="font-size:0.8em; background-color:#dc3545; color:white; padding: 2px 5px; border-radius:3px;">Estático</span>' if model_name == 'ARIMA' else ''
        html_table += f"""<tr style="{row_style}"><td style="padding: 8px; border-bottom:1px solid #ddd;">{model_name}{tag}{static_tag}</td><td style="padding: 8px; border-bottom:1px solid #ddd; text-align: right;">{prediction:,.2f}</td></tr>"""
    html_table += "</tbody></table></div>"
    return html_table

def create_prediction_graph(historic_data_series, preds, asset, precio_anterior):
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(9, 5)); historic_data = historic_data_series.tail(100)
    ax.plot(historic_data.index, historic_data.values, label=f'Histórico de {asset}', color='black', alpha=0.7)
    last_date = historic_data.index[-1]; prediction_date = last_date + pd.Timedelta(days=1)
    colors = {'ARIMA': 'red', 'Regresión Lineal': 'orange', 'Gradient Boosting': 'purple', 'Random Forest': 'blue'}
    best_model_name = "Random Forest"
    for model_name, prediction in preds.items():
        is_best = (model_name == best_model_name)
        ax.plot([last_date, prediction_date], [precio_anterior, prediction], linestyle='--', marker='o', color=colors[model_name], label=f'{model_name}: {prediction:,.2f}', linewidth=3.0 if is_best else 1.5, alpha=1.0 if is_best else 0.8)
    ax.set_title(f'Pronósticos para {asset} bajo Escenario Simulado', fontsize=16); ax.set_ylabel('Precio (USD)'); ax.legend(loc='best'); plt.tight_layout(); plt.show()

def on_asset_change_simulador(change):
    asset = change['new']
    last_price = df_final[f'{asset}_Close'].iloc[-1]
    precio_anterior_slider.min = last_price * 0.8; precio_anterior_slider.max = last_price * 1.2
    precio_anterior_slider.value = last_price

asset_selector_sim.observe(on_asset_change_simulador, names='value')

tab_content = [Output() for _ in range(4)]
tabs = widgets.Tab(children=tab_content)
tabs.set_title(0, '1. Impacto Tasas Interés'); tabs.set_title(1, '2. Impacto Inflación'); tabs.set_title(2, '3. Impacto Datos Históricos'); tabs.set_title(3, '4. DEMO INTERACTIVA')
asset_selector_static = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Activo para Análisis:')

def update_static_tabs(change):
    asset = change['new']
    with tab_content[0]: tab_content[0].clear_output(wait=True); display(create_tab_objective1(asset))
    with tab_content[1]: tab_content[1].clear_output(wait=True); display(create_tab_objective2(asset))
    with tab_content[2]: tab_content[2].clear_output(wait=True); display(create_tab_objective3(asset))

asset_selector_static.observe(update_static_tabs, names='value')

with tab_content[3]:
    controles_sliders = VBox([ HBox([tasa_interes_slider, tasa_interes_text]), HBox([inflacion_slider, inflacion_text]), HBox([vix_slider, vix_text]), HBox([precio_anterior_slider, precio_anterior_text]) ])
    controles_principales = VBox([asset_selector_sim, HTML("<hr><p><b>Defina un escenario:</b></p>"), controles_sliders])
    resultados_ui = HBox([table_output_container, graph_output_container], layout=Layout(align_items='flex-start', justify_content='space-between'))
    interactive_sim = widgets.interactive_output(update_simulador, {'asset': asset_selector_sim, 'tasa_interes': tasa_interes_slider, 'inflacion': inflacion_slider, 'vix': vix_slider, 'precio_anterior': precio_anterior_slider})
    display(VBox([HTML("<h3>Simulador de Escenarios en Vivo</h3>"), controles_principales, resultados_ui]))

display(HTML("<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las pestañas 1-3. La pestaña 4 tiene su propio selector.</p>")); display(asset_selector_static); display(tabs)
update_static_tabs({'new': 'AMZN'})
on_asset_change_simulador({'new': asset_selector_sim.value})

✅ Datos cargados correctamente.
✅ Ingeniería de características completada.
✅ Todos los modelos han sido entrenados y evaluados.


HTML(value='<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las…

Dropdown(description='Activo para Análisis:', options=('AMZN', 'GOOGL'), value='AMZN')

In [ ]:
# ==============================================================================
# PASO 1: INSTALACIÓN Y CARGA DE LIBRERÍAS Y DATOS
# ==============================================================================
print("Instalando librerías necesarias...")
# !pip install pandas scikit-learn statsmodels ipywidgets matplotlib shap -q
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from statsmodels.tsa.arima.model import ARIMA
import ipywidgets as widgets
from ipywidgets import VBox, HBox, HTML, Output, link, Layout
import matplotlib.pyplot as plt
import shap
import warnings

warnings.filterwarnings('ignore')

# Cargar los datos
url = 'https://raw.githubusercontent.com/jaimeisaac2020/Python-analsisis-basicos/mi-github/dataset_completo_google_amazon.csv'
df = pd.read_csv(url, index_col='Date', parse_dates=True)
print("✅ Datos cargados correctamente.")

# ==============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS
# ==============================================================================
df_eng = df.copy()
for asset in ['GOOGL', 'AMZN']:
    df_eng[f'{asset}_Return_1d'] = df_eng[f'{asset}_Close'].pct_change()
    df_eng[f'{asset}_Volatility_20'] = df_eng[f'{asset}_Return_1d'].rolling(window=20).std()
    for lag in range(1, 6): df_eng[f'{asset}_Close_Lag_{lag}'] = df_eng[f'{asset}_Close'].shift(lag)

df_final = df_eng.dropna()
print("✅ Ingeniería de características completada.")

# ==============================================================================
# PASO 3: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS
# ==============================================================================
results = {}
models = {}
X_train_dict = {}
X_test_dict = {}
features_base_all = ['VIX', 'SP500', 'NASDAQ', 'Treasury_10Y', 'Inflacion_T5YIE']

for asset in ['GOOGL', 'AMZN']:
    results[asset] = {}
    models[asset] = {}

    train_size = int(len(df_final) * 0.8)
    train_df, test_df = df_final.iloc[:train_size], df_final.iloc[train_size:]
    y_train, y_test = train_df[f'{asset}_Close'], test_df[f'{asset}_Close']

    features_macro_base = ['VIX', 'SP500', 'NASDAQ']
    rf_base = RandomForestRegressor(random_state=42).fit(train_df[features_macro_base], y_train); preds = rf_base.predict(test_df[features_macro_base])
    results[asset]['RF_Base_Macro'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_tasa = features_macro_base + ['Treasury_10Y']
    rf_con_tasa = RandomForestRegressor(random_state=42).fit(train_df[features_con_tasa], y_train); preds = rf_con_tasa.predict(test_df[features_con_tasa])
    results[asset]['RF_Con_Tasa'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_inflacion = features_con_tasa + ['Inflacion_T5YIE']
    rf_con_inf = RandomForestRegressor(random_state=42).fit(train_df[features_con_inflacion], y_train); preds = rf_con_inf.predict(test_df[features_con_inflacion])
    results[asset]['RF_Con_Inflacion'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
    X_train_enriched, y_train_enriched = train_df[features_enriched], train_df[f'{asset}_Close']
    X_test_enriched, y_test_enriched = test_df[features_enriched], test_df[f'{asset}_Close']
    X_train_dict[asset], X_test_dict[asset] = X_train_enriched, X_test_enriched

    models[asset]['Random Forest'] = RandomForestRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Gradient Boosting'] = GradientBoostingRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Regresión Lineal'] = LinearRegression().fit(X_train_enriched, y_train_enriched)

    preds_enriched = models[asset]['Random Forest'].predict(X_test_enriched)
    results[asset]['RF_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, preds_enriched), 'R2': r2_score(y_test_enriched, preds_enriched)}

    arima_order = (1, 1, 1) if asset == 'GOOGL' else (0, 1, 0)
    arima_model = ARIMA(y_train_enriched, order=arima_order).fit()
    models[asset]['ARIMA_pred_estatica'] = arima_model.forecast(steps=1).iloc[0]

print("✅ Todos los modelos han sido entrenados y evaluados.")

# ==============================================================================
# PASO 4: FUNCIONES PARA CREAR EL CONTENIDO DE CADA PESTAÑA
# ==============================================================================
def create_tab_exploration():
    head_html = HTML("<h3>Primeros 5 registros (Head)</h3>" + df.head().to_html())
    tail_html = HTML("<h3>Últimos 5 registros (Tail)</h3>" + df.tail().to_html())
    plots_out = Output();
    with plots_out:
        display(HTML("<h4>Gráficos de Series de Tiempo Individuales</h4>"))
        for col in df.columns:
            if col not in ['GOOGL_Return_1d', 'AMZN_Return_1d', 'GOOGL_Volatility_20', 'AMZN_Volatility_20']:
                fig, ax = plt.subplots(figsize=(8, 2)); df[col].plot(ax=ax, title=col); plt.tight_layout(); plt.show()
    return VBox([head_html, tail_html, plots_out])

def create_tab_objective1(asset):
    res_base = results[asset]['RF_Base_Macro']; res_completo = results[asset]['RF_Con_Tasa']
    df_res = pd.DataFrame([res_base, res_completo], index=['Modelo Base', 'Modelo con Tasa Int.'])
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue', label=f'Precio {asset}'); ax1.set_ylabel(f'Precio {asset} (USD)', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Treasury_10Y'], color='red', linestyle='--', label='Tasa 10Y'); ax2.set_ylabel('Tasa Treasury 10Y (%)', color='red'); ax2.tick_params(axis='y', labelcolor='red'); plt.title(f'Contexto: Precio de {asset} vs. Tasa de Interés'); plt.show()
    results_plot_out = Output()
    with results_plot_out:
        fig, ax = plt.subplots(figsize=(5, 3)); bars = ax.bar(df_res.index, df_res['MSE'], color=['gray', 'skyblue']); ax.set_title(f'Reducción de Error (MSE)'); ax.bar_label(bars, fmt='{:,.2f}'); plt.show()
    return VBox([HTML("<h4>Validación H2: Impacto de Tasas de Interés</h4>"), context_plot_out, HBox([styled_table_html, results_plot_out])])

def create_tab_objective2(asset):
    res_sin = results[asset]['RF_Con_Tasa']; res_con = results[asset]['RF_Con_Inflacion']
    df_res = pd.DataFrame([res_sin, res_con], index=['Modelo SIN Inflación', 'Modelo CON Inflación'])
    incremento_error = (res_sin['MSE'] / res_con['MSE'] - 1) * 100 if res_con['MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue', label=f'Precio {asset}'); ax1.set_ylabel(f'Precio {asset} (USD)', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Inflacion_T5YIE'], color='green', linestyle='--', label='Inflación 5A'); ax2.set_ylabel('Expectativa Inflación 5A (%)', color='green'); ax2.tick_params(axis='y', labelcolor='green'); plt.title(f'Contexto: Precio de {asset} vs. Expectativas de Inflación'); plt.show()
    return VBox([HTML("<h4>Validación H3: Impacto de la Inflación</h4>"), context_plot_out, styled_table_html, HTML(f"<div style='margin-top:10px; padding:10px; background-color:#f8d7da; border-radius:5px;'>Eliminar la inflación <b>aumentó el error en un {incremento_error:.2f}%</b> para {asset}.</div>")])

def create_tab_objective3(asset):
    res_base = results[asset]['RF_Base_Macro']; res_enriquecido = results[asset]['RF_Enriquecido']
    df_res = pd.DataFrame([res_base, res_enriquecido], index=['Modelo SIN Históricos', 'Modelo CON Históricos'])
    r2_color = 'color:red' if df_res.loc['Modelo CON Históricos', 'R2'] < 0 else 'color:green'
    html_resumen = (f"<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>¡Resultado Clave para {asset}!</b> El R² pasó de {res_base['R2']:.2f} a <b style='{r2_color}'>{res_enriquecido['R2']:.2f}</b>.</div>") if asset == 'AMZN' else ""
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())
    return VBox([HTML("<h4>Validación H4: Impacto de Datos Históricos</h4>"), HTML(html_resumen), styled_table_html])

def create_tab_interpretability(asset):
    model = models[asset]['Random Forest']; X_test = X_test_dict[asset]; X_train = X_train_dict[asset]
    shap_out = Output()
    with shap_out:
        explainer = shap.TreeExplainer(model); shap_values = explainer(X_test); plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_values, X_test, show=False, plot_size=None); plt.title(f'SHAP Summary Plot para {asset}'); plt.tight_layout(); plt.show()
    pdp_out = Output()
    with pdp_out:
        features_to_plot = ['Treasury_10Y', 'Inflacion_T5YIE', 'VIX', f'{asset}_Close_Lag_1']
        fig, ax = plt.subplots(1, len(features_to_plot), figsize=(15, 3))
        PartialDependenceDisplay.from_estimator(model, X_train, features_to_plot, ax=ax); plt.suptitle(f'Partial Dependence Plots (PDP) para {asset}'); plt.tight_layout(); plt.show()
    return VBox([HTML("<h3>Interpretabilidad del Modelo Random Forest</h3>"), HTML("<h4>Gráfico SHAP: Importancia y Dirección de Variables</h4><p>Rojo=valor alto, Azul=valor bajo. Derecha=sube predicción, Izquierda=baja predicción.</p>"), shap_out, HTML("<hr><h4>Gráficos PDP: Relación Promedio de Variables Clave</h4><p>Muestra el efecto promedio de una variable en la predicción, aislando su impacto del resto.</p>"), pdp_out])

def create_tab_simulador():
    asset_selector_sim = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Seleccione el Activo:')
    tasa_interes_slider = widgets.FloatSlider(value=4.0, min=2.0, max=6.0, step=0.05, description='Tasa Interés:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    inflacion_slider = widgets.FloatSlider(value=2.5, min=1.5, max=4.0, step=0.05, description='Inflación:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    vix_slider = widgets.FloatSlider(value=15, min=10, max=40, step=1, description='VIX:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    precio_anterior_slider = widgets.FloatSlider(value=df_final['AMZN_Close'].iloc[-1], min=150, max=250, step=0.5, description='Precio Anterior:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    tasa_interes_text, inflacion_text, vix_text, precio_anterior_text = (widgets.FloatText(layout=Layout(width='20%')) for _ in range(4))
    link((tasa_interes_slider, 'value'), (tasa_interes_text, 'value')); link((inflacion_slider, 'value'), (inflacion_text, 'value')); link((vix_slider, 'value'), (vix_text, 'value')); link((precio_anterior_slider, 'value'), (precio_anterior_text, 'value'))
    table_output_container = HTML(); graph_output_container = Output()

    def update_simulador(asset, tasa_interes, inflacion, vix, precio_anterior):
        if any(v is None for v in [tasa_interes, inflacion, vix, precio_anterior]): return
        last_known = df_final.iloc[-1].copy(); input_data = pd.DataFrame([last_known])
        input_data['Treasury_10Y'], input_data['Inflacion_T5YIE'], input_data['VIX'], input_data[f'{asset}_Close_Lag_1'] = tasa_interes, inflacion, vix, precio_anterior
        for lag in range(2, 6): input_data[f'{asset}_Close_Lag_{lag}'] = precio_anterior
        features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
        input_data_final = input_data[features_enriched]
        preds = {name: model.predict(input_data_final)[0] for name, model in models[asset].items() if name != 'ARIMA_pred_estatica'}; preds['ARIMA'] = models[asset]['ARIMA_pred_estatica']
        table_output_container.value = create_html_table(preds, precio_anterior)
        graph_output_container.clear_output(wait=True)
        with graph_output_container: create_prediction_graph(df_final[f'{asset}_Close'], preds, asset, precio_anterior)

    def create_html_table(preds, precio_anterior):
        best_model_name = "Random Forest"
        html = f"""<div style="border:1px solid #ccc;padding:15px;border-radius:8px;"><h3 style="margin-top:0;">Tabla de Predicciones</h3><p style="font-size:0.9em;">Precio Anterior: <b>{precio_anterior:,.2f} USD</b></p><table style="width:100%; border-collapse: collapse;"><thead><tr style="background-color:#005a9c;color:white;"><th style="padding:10px;text-align:left;">Modelo</th><th style="padding:10px;text-align:right;">Precio Predicho</th></tr></thead><tbody>"""
        for model_name in ['Random Forest', 'Gradient Boosting', 'Regresión Lineal', 'ARIMA']:
            pred = preds[model_name]; is_best = (model_name == best_model_name); style = 'background-color:#d4edda;color:#155724;font-weight:bold;' if is_best else ''
            tag = ' <span style="font-size:0.8em;background-color:#28a745;color:white;padding:2px 5px;border-radius:3px;">Mejor Modelo</span>' if is_best else ''
            static_tag = ' <span style="font-size:0.8em;background-color:#dc3545;color:white;padding:2px 5px;border-radius:3px;">Estático</span>' if model_name == 'ARIMA' else ''
            html += f"""<tr style="{style}"><td style="padding:8px;border-bottom:1px solid #ddd;">{model_name}{tag}{static_tag}</td><td style="padding:8px;border-bottom:1px solid #ddd;text-align:right;">{pred:,.2f}</td></tr>"""
        return html + "</tbody></table></div>"

    def create_prediction_graph(historic_data_series, preds, asset, precio_anterior):
        plt.style.use('seaborn-v0_8-whitegrid'); fig, ax = plt.subplots(figsize=(9, 5)); historic_data = historic_data_series.tail(100)
        ax.plot(historic_data.index, historic_data.values, label=f'Histórico de {asset}', color='black', alpha=0.7)
        last_date = historic_data.index[-1]; prediction_date = last_date + pd.Timedelta(days=1); colors = {'ARIMA': 'red', 'Regresión Lineal': 'orange', 'Gradient Boosting': 'purple', 'Random Forest': 'blue'}; best_model_name = "Random Forest"
        for model_name, prediction in preds.items():
            is_best = (model_name == best_model_name)
            ax.plot([last_date, prediction_date], [precio_anterior, prediction], linestyle='--', marker='o', color=colors[model_name], label=f'{model_name}: {prediction:,.2f}', linewidth=3.0 if is_best else 1.5, alpha=1.0 if is_best else 0.8)
        ax.set_title(f'Pronósticos para {asset} bajo Escenario Simulado', fontsize=16); ax.set_ylabel('Precio (USD)'); ax.legend(loc='best'); plt.tight_layout(); plt.show()

    def on_asset_change_simulador(change):
        asset = change['new']; last_price = df_final[f'{asset}_Close'].iloc[-1]
        precio_anterior_slider.min = last_price * 0.8; precio_anterior_slider.max = last_price * 1.2
        precio_anterior_slider.value = last_price

    asset_selector_sim.observe(on_asset_change_simulador, names='value')
    controles_sliders = VBox([HBox([tasa_interes_slider, tasa_interes_text]), HBox([inflacion_slider, inflacion_text]), HBox([vix_slider, vix_text]), HBox([precio_anterior_slider, precio_anterior_text])])
    controles_principales = VBox([asset_selector_sim, HTML("<hr><p><b>Defina un escenario:</b></p>"), controles_sliders])
    resultados_ui = HBox([table_output_container, graph_output_container], layout=Layout(align_items='flex-start', justify_content='space-between'))
    widgets.interactive_output(update_simulador, {'asset': asset_selector_sim, 'tasa_interes': tasa_interes_slider, 'inflacion': inflacion_slider, 'vix': vix_slider, 'precio_anterior': precio_anterior_slider})
    on_asset_change_simulador({'new': asset_selector_sim.value})
    return VBox([HTML("<h3>Simulador de Escenarios en Vivo</h3>"), controles_principales, resultados_ui])

# --- Montaje Final del Dashboard ---
tab_children = [Output() for _ in range(6)]
tabs = widgets.Tab(children=tab_children)
tabs.set_title(0, '0. Exploración Datos'); tabs.set_title(1, '1. Impacto Tasas Int.'); tabs.set_title(2, '2. Impacto Inflación'); tabs.set_title(3, '3. Impacto D. Históricos'); tabs.set_title(4, '4. Interpretabilidad'); tabs.set_title(5, '5. DEMO INTERACTIVA')
asset_selector_static = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Activo para Análisis:')

def update_all_tabs(change):
    asset = change['new']
    with tab_children[0]: tab_children[0].clear_output(wait=True); display(create_tab_exploration())
    with tab_children[1]: tab_children[1].clear_output(wait=True); display(create_tab_objective1(asset))
    with tab_children[2]: tab_children[2].clear_output(wait=True); display(create_tab_objective2(asset))
    with tab_children[3]: tab_children[3].clear_output(wait=True); display(create_tab_objective3(asset))
    with tab_children[4]: tab_children[4].clear_output(wait=True); display(create_tab_interpretability(asset))

asset_selector_static.observe(update_all_tabs, names='value')

with tab_children[5]:
    tab_children[5].clear_output(wait=True)
    display(create_tab_simulador())

display(HTML("<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las pestañas 0-4. La pestaña 5 (DEMO) tiene su propio selector.</p>")); display(asset_selector_static); display(tabs)
update_all_tabs({'new': 'AMZN'})

Instalando librerías necesarias...
✅ Datos cargados correctamente.
✅ Ingeniería de características completada.
✅ Todos los modelos han sido entrenados y evaluados.


HTML(value='<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las…

Dropdown(description='Activo para Análisis:', options=('AMZN', 'GOOGL'), value='AMZN')

In [ ]:
# ==============================================================================
# PASO 1: INSTALACIÓN Y CARGA DE LIBRERÍAS Y DATOS
# ==============================================================================
print("Instalando librerías necesarias...")
# !pip install pandas scikit-learn statsmodels ipywidgets matplotlib shap -q
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from statsmodels.tsa.arima.model import ARIMA
import ipywidgets as widgets
from ipywidgets import VBox, HBox, HTML, Output, link, Layout
import matplotlib.pyplot as plt
import shap
import warnings

warnings.filterwarnings('ignore')

# Cargar los datos
url = 'https://raw.githubusercontent.com/jaimeisaac2020/Python-analsisis-basicos/mi-github/dataset_completo_google_amazon.csv'
df = pd.read_csv(url, index_col='Date', parse_dates=True)
print("✅ Datos cargados correctamente.")

# ==============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS
# ==============================================================================
df_eng = df.copy()
for asset in ['GOOGL', 'AMZN']:
    df_eng[f'{asset}_Return_1d'] = df_eng[f'{asset}_Close'].pct_change()
    df_eng[f'{asset}_Volatility_20'] = df_eng[f'{asset}_Return_1d'].rolling(window=20).std()
    for lag in range(1, 6): df_eng[f'{asset}_Close_Lag_{lag}'] = df_eng[f'{asset}_Close'].shift(lag)

df_final = df_eng.dropna()
print("✅ Ingeniería de características completada.")

# ==============================================================================
# PASO 3: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS
# ==============================================================================
results = {}
models = {}
X_train_dict = {}
X_test_dict = {}
features_base_all = ['VIX', 'SP500', 'NASDAQ', 'Treasury_10Y', 'Inflacion_T5YIE']
arima_predictions = {}

for asset in ['GOOGL', 'AMZN']:
    results[asset] = {}
    models[asset] = {}

    train_size = int(len(df_final) * 0.8)
    train_df, test_df = df_final.iloc[:train_size], df_final.iloc[train_size:]
    y_train, y_test = train_df[f'{asset}_Close'], test_df[f'{asset}_Close']

    features_macro_base = ['VIX', 'SP500', 'NASDAQ']
    rf_base = RandomForestRegressor(random_state=42).fit(train_df[features_macro_base], y_train); preds = rf_base.predict(test_df[features_macro_base])
    results[asset]['RF_Base_Macro'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_tasa = features_macro_base + ['Treasury_10Y']
    rf_con_tasa = RandomForestRegressor(random_state=42).fit(train_df[features_con_tasa], y_train); preds = rf_con_tasa.predict(test_df[features_con_tasa])
    results[asset]['RF_Con_Tasa'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_inflacion = features_con_tasa + ['Inflacion_T5YIE']
    rf_con_inf = RandomForestRegressor(random_state=42).fit(train_df[features_con_inflacion], y_train); preds = rf_con_inf.predict(test_df[features_con_inflacion])
    results[asset]['RF_Con_Inflacion'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
    X_train_enriched, y_train_enriched = train_df[features_enriched], train_df[f'{asset}_Close']
    X_test_enriched, y_test_enriched = test_df[features_enriched], test_df[f'{asset}_Close']
    X_train_dict[asset], X_test_dict[asset] = X_train_enriched, X_test_enriched

    models[asset]['Random Forest'] = RandomForestRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Gradient Boosting'] = GradientBoostingRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Regresión Lineal'] = LinearRegression().fit(X_train_enriched, y_train_enriched)

    for name, model in models[asset].items():
        preds = model.predict(X_test_enriched)
        results[asset][f'{name}_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, preds), 'R2': r2_score(y_test_enriched, preds)}

    arima_order = (1, 1, 1) if asset == 'GOOGL' else (0, 1, 0)
    arima_model = ARIMA(y_train_enriched, order=arima_order).fit()
    arima_predictions[asset] = arima_model.forecast(steps=len(y_test_enriched))
    results[asset]['ARIMA_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, arima_predictions[asset]), 'R2': r2_score(y_test_enriched, arima_predictions[asset])}
    models[asset]['ARIMA_pred_estatica'] = arima_predictions[asset].iloc[0]

print("✅ Todos los modelos han sido entrenados y evaluados.")

# ==============================================================================
# PASO 4: FUNCIONES PARA CREAR EL CONTENIDO DE CADA PESTAÑA
# ==============================================================================

def create_tab_exploration():
    # (código sin cambios)
    head_html = HTML("<h3>Primeros 5 registros (Head)</h3>" + df.head().to_html())
    tail_html = HTML("<h3>Últimos 5 registros (Tail)</h3>" + df.tail().to_html())
    plots_out = Output();
    with plots_out:
        display(HTML("<h4>Gráficos de Series de Tiempo Individuales</h4>"))
        for col in df.columns:
            if col not in ['GOOGL_Return_1d', 'AMZN_Return_1d', 'GOOGL_Volatility_20', 'AMZN_Volatility_20']:
                fig, ax = plt.subplots(figsize=(8, 2)); df[col].plot(ax=ax, title=col); plt.tight_layout(); plt.show()
    return VBox([head_html, tail_html, plots_out])

def create_tab_objective1(asset):
    # (código sin cambios)
    res_base = results[asset]['RF_Base_Macro']; res_completo = results[asset]['RF_Con_Tasa']
    df_res = pd.DataFrame([res_base, res_completo], index=['Modelo Base', 'Modelo con Tasa Int.'])
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue', label=f'Precio {asset}'); ax1.set_ylabel(f'Precio {asset} (USD)', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Treasury_10Y'], color='red', linestyle='--', label='Tasa 10Y'); ax2.set_ylabel('Tasa Treasury 10Y (%)', color='red'); ax2.tick_params(axis='y', labelcolor='red'); plt.title(f'Contexto: Precio de {asset} vs. Tasa de Interés'); plt.show()
    results_plot_out = Output()
    with results_plot_out:
        fig, ax = plt.subplots(figsize=(5, 3)); bars = ax.bar(df_res.index, df_res['MSE'], color=['gray', 'skyblue']); ax.set_title(f'Reducción de Error (MSE)'); ax.bar_label(bars, fmt='{:,.2f}'); plt.show()
    return VBox([HTML("<h4>Validación H2: Impacto de Tasas de Interés</h4>"), context_plot_out, HBox([styled_table_html, results_plot_out])])

def create_tab_objective2(asset):
    # (código sin cambios)
    res_sin = results[asset]['RF_Con_Tasa']; res_con = results[asset]['RF_Con_Inflacion']
    df_res = pd.DataFrame([res_sin, res_con], index=['Modelo SIN Inflación', 'Modelo CON Inflación'])
    incremento_error = (res_sin['MSE'] / res_con['MSE'] - 1) * 100 if res_con['MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue', label=f'Precio {asset}'); ax1.set_ylabel(f'Precio {asset} (USD)', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Inflacion_T5YIE'], color='green', linestyle='--', label='Inflación 5A'); ax2.set_ylabel('Expectativa Inflación 5A (%)', color='green'); ax2.tick_params(axis='y', labelcolor='green'); plt.title(f'Contexto: Precio de {asset} vs. Expectativas de Inflación'); plt.show()
    return VBox([HTML("<h4>Validación H3: Impacto de la Inflación</h4>"), context_plot_out, styled_table_html, HTML(f"<div style='margin-top:10px; padding:10px; background-color:#f8d7da; border-radius:5px;'>Eliminar la inflación <b>aumentó el error en un {incremento_error:.2f}%</b> para {asset}.</div>")])

def create_tab_objective3(asset):
    # (código sin cambios)
    res_base = results[asset]['RF_Base_Macro']; res_enriquecido = results[asset]['RF_Enriquecido']
    df_res = pd.DataFrame([res_base, res_enriquecido], index=['Modelo SIN Históricos', 'Modelo CON Históricos'])
    r2_color = 'color:red' if df_res.loc['Modelo CON Históricos', 'R2'] < 0 else 'color:green'
    html_resumen = (f"<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>¡Resultado Clave para {asset}!</b> El R² pasó de {res_base['R2']:.2f} a <b style='{r2_color}'>{res_enriquecido['R2']:.2f}</b>.</div>") if asset == 'AMZN' else ""
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())
    return VBox([HTML("<h4>Validación H4: Impacto de Datos Históricos</h4>"), HTML(html_resumen), styled_table_html])

# --- NUEVA PESTAÑA: Validación de Hipótesis 1 ---
def create_tab_h1_validation(asset):
    asset_results = {
        'ARIMA': results[asset]['ARIMA_Enriquecido'],
        'Random Forest': results[asset]['Random Forest_Enriquecido'],
        'Gradient Boosting': results[asset]['Gradient Boosting_Enriquecido']
    }
    df_res = pd.DataFrame(asset_results).T

    reduction_vs_arima = (1 - df_res.loc['Random Forest', 'MSE'] / df_res.loc['ARIMA', 'MSE']) * 100

    styled_table_html = HTML(df_res.style.format('{:.4f}')
                             .highlight_min(subset='MSE', color='#d4edda')
                             .applymap(lambda x: 'color:red' if x < 0 else 'color:green', subset=['R2']).to_html())

    graph_out = Output()
    with graph_out:
        fig, ax = plt.subplots(figsize=(10, 5))
        test_dates = test_df.index
        ax.plot(test_dates, test_df[f'{asset}_Close'], label='Precio Real', color='black', linewidth=2)
        ax.plot(test_dates, arima_predictions[asset], label='ARIMA (Tradicional)', color='red', linestyle='--')
        ax.plot(test_dates, models[asset]['Random Forest'].predict(X_test_dict[asset]), label='Random Forest (Supervisado)', color='blue', linestyle='--')
        ax.set_title(f'Comparación Visual de Predicciones para {asset}')
        ax.legend(); plt.tight_layout(); plt.show()

    return VBox([
        HTML("<h3>Validación Hipótesis 1: Superioridad de Modelos Supervisados</h3>"),
        HTML("<p>Esta hipótesis, ligada al <b>Objetivo Específico 4</b>, postula que los modelos supervisados son más precisos que los tradicionales como ARIMA.</p>"),
        HTML("<hr><h4>Evidencia Cuantitativa</h4>"),
        styled_table_html,
        HTML(f"<div style='margin-top:10px; padding:10px; background-color:#d4edda; border-radius:5px;'><b>Conclusión Cuantitativa:</b> Random Forest reduce el error (MSE) en un <b>{reduction_vs_arima:.2f}%</b> en comparación con ARIMA para {asset}.</div>"),
        HTML("<hr><h4>Evidencia Visual</h4>"),
        graph_out,
        HTML("<p><b>Interpretación Visual:</b> El gráfico muestra claramente cómo el modelo ARIMA (rojo) falla en capturar la tendencia, mientras que Random Forest (azul) se adapta a la dinámica del precio real.</p>"),
        HTML("<hr><h3 style='color:green;'>Veredicto: Hipótesis 1 Aceptada</h3>")
    ])

def create_tab_interpretability(asset):
    # (código sin cambios)
    model = models[asset]['Random Forest']; X_test = X_test_dict[asset]; X_train = X_train_dict[asset]
    shap_out = Output()
    with shap_out:
        explainer = shap.TreeExplainer(model); shap_values = explainer(X_test); plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_values, X_test, show=False, plot_size=None); plt.title(f'SHAP Summary Plot para {asset}'); plt.tight_layout(); plt.show()
    pdp_out = Output()
    with pdp_out:
        features_to_plot = ['Treasury_10Y', 'Inflacion_T5YIE', 'VIX', f'{asset}_Close_Lag_1']
        fig, ax = plt.subplots(1, len(features_to_plot), figsize=(15, 3))
        PartialDependenceDisplay.from_estimator(model, X_train, features_to_plot, ax=ax); plt.suptitle(f'Partial Dependence Plots (PDP) para {asset}'); plt.tight_layout(); plt.show()
    return VBox([HTML("<h3>Interpretabilidad del Modelo Random Forest</h3>"), HTML("<h4>Gráfico SHAP</h4>"), shap_out, HTML("<hr><h4>Gráficos PDP</h4>"), pdp_out])

def create_tab_simulador():
    # (código sin cambios)
    asset_selector_sim = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Seleccione el Activo:')
    tasa_interes_slider = widgets.FloatSlider(value=4.0, min=2.0, max=6.0, step=0.05, description='Tasa Interés:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    inflacion_slider = widgets.FloatSlider(value=2.5, min=1.5, max=4.0, step=0.05, description='Inflación:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    vix_slider = widgets.FloatSlider(value=15, min=10, max=40, step=1, description='VIX:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    precio_anterior_slider = widgets.FloatSlider(value=df_final['AMZN_Close'].iloc[-1], min=150, max=250, step=0.5, description='Precio Anterior:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    tasa_interes_text, inflacion_text, vix_text, precio_anterior_text = (widgets.FloatText(layout=Layout(width='20%')) for _ in range(4))
    link((tasa_interes_slider, 'value'), (tasa_interes_text, 'value')); link((inflacion_slider, 'value'), (inflacion_text, 'value')); link((vix_slider, 'value'), (vix_text, 'value')); link((precio_anterior_slider, 'value'), (precio_anterior_text, 'value'))
    table_output_container = HTML(); graph_output_container = Output()

    # Funciones internas del simulador
    def update_simulador(asset, tasa_interes, inflacion, vix, precio_anterior):
        if any(v is None for v in [tasa_interes, inflacion, vix, precio_anterior]): return
        last_known = df_final.iloc[-1].copy(); input_data = pd.DataFrame([last_known])
        input_data['Treasury_10Y'], input_data['Inflacion_T5YIE'], input_data['VIX'], input_data[f'{asset}_Close_Lag_1'] = tasa_interes, inflacion, vix, precio_anterior
        for lag in range(2, 6): input_data[f'{asset}_Close_Lag_{lag}'] = precio_anterior
        features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
        input_data_final = input_data[features_enriched]
        preds = {name: model.predict(input_data_final)[0] for name, model in models[asset].items() if name != 'ARIMA_pred_estatica'}; preds['ARIMA'] = models[asset]['ARIMA_pred_estatica']
        table_output_container.value = create_html_table(preds, precio_anterior)
        graph_output_container.clear_output(wait=True)
        with graph_output_container: create_prediction_graph(df_final[f'{asset}_Close'], preds, asset, precio_anterior)

    def create_html_table(preds, precio_anterior):
        best_model_name = "Random Forest"
        html = f"""<div style="border:1px solid #ccc;padding:15px;border-radius:8px;"><h3 style="margin-top:0;">Tabla de Predicciones</h3><p style="font-size:0.9em;">Precio Anterior: <b>{precio_anterior:,.2f} USD</b></p><table style="width:100%; border-collapse: collapse;"><thead><tr style="background-color:#005a9c;color:white;"><th style="padding:10px;text-align:left;">Modelo</th><th style="padding:10px;text-align:right;">Precio Predicho</th></tr></thead><tbody>"""
        for model_name in ['Random Forest', 'Gradient Boosting', 'Regresión Lineal', 'ARIMA']:
            pred = preds[model_name]; is_best = (model_name == best_model_name); style = 'background-color:#d4edda;color:#155724;font-weight:bold;' if is_best else ''
            tag = ' <span style="font-size:0.8em;background-color:#28a745;color:white;padding:2px 5px;border-radius:3px;">Mejor Modelo</span>' if is_best else ''
            static_tag = ' <span style="font-size:0.8em;background-color:#dc3545;color:white;padding:2px 5px;border-radius:3px;">Estático</span>' if model_name == 'ARIMA' else ''
            html += f"""<tr style="{style}"><td style="padding:8px;border-bottom:1px solid #ddd;">{model_name}{tag}{static_tag}</td><td style="padding:8px;border-bottom:1px solid #ddd;text-align:right;">{pred:,.2f}</td></tr>"""
        return html + "</tbody></table></div>"

    def create_prediction_graph(historic_data_series, preds, asset, precio_anterior):
        plt.style.use('seaborn-v0_8-whitegrid'); fig, ax = plt.subplots(figsize=(9, 5)); historic_data = historic_data_series.tail(100)
        ax.plot(historic_data.index, historic_data.values, label=f'Histórico de {asset}', color='black', alpha=0.7)
        last_date = historic_data.index[-1]; prediction_date = last_date + pd.Timedelta(days=1); colors = {'ARIMA': 'red', 'Regresión Lineal': 'orange', 'Gradient Boosting': 'purple', 'Random Forest': 'blue'}; best_model_name = "Random Forest"
        for model_name, prediction in preds.items():
            is_best = (model_name == best_model_name)
            ax.plot([last_date, prediction_date], [precio_anterior, prediction], linestyle='--', marker='o', color=colors[model_name], label=f'{model_name}: {prediction:,.2f}', linewidth=3.0 if is_best else 1.5, alpha=1.0 if is_best else 0.8)
        ax.set_title(f'Pronósticos para {asset} bajo Escenario Simulado', fontsize=16); ax.set_ylabel('Precio (USD)'); ax.legend(loc='best'); plt.tight_layout(); plt.show()

    def on_asset_change_simulador(change):
        asset = change['new']; last_price = df_final[f'{asset}_Close'].iloc[-1]
        precio_anterior_slider.min = last_price * 0.8; precio_anterior_slider.max = last_price * 1.2
        precio_anterior_slider.value = last_price

    asset_selector_sim.observe(on_asset_change_simulador, names='value')
    controles_sliders = VBox([HBox([tasa_interes_slider, tasa_interes_text]), HBox([inflacion_slider, inflacion_text]), HBox([vix_slider, vix_text]), HBox([precio_anterior_slider, precio_anterior_text])])
    controles_principales = VBox([asset_selector_sim, HTML("<hr><p><b>Defina un escenario:</b></p>"), controles_sliders])
    resultados_ui = HBox([table_output_container, graph_output_container], layout=Layout(align_items='flex-start', justify_content='space-between'))
    widgets.interactive_output(update_simulador, {'asset': asset_selector_sim, 'tasa_interes': tasa_interes_slider, 'inflacion': inflacion_slider, 'vix': vix_slider, 'precio_anterior': precio_anterior_slider})
    on_asset_change_simulador({'new': asset_selector_sim.value})
    return VBox([HTML("<h3>Simulador de Escenarios en Vivo</h3>"), controles_principales, resultados_ui])

# --- Montaje Final del Dashboard ---
tab_children = [Output() for _ in range(7)]
tabs = widgets.Tab(children=tab_children)
tabs.set_title(0, '0. Exploración Datos'); tabs.set_title(1, '1. Validación H1'); tabs.set_title(2, '2. Impacto Tasas Int.'); tabs.set_title(3, '3. Impacto Inflación'); tabs.set_title(4, '4. Impacto D. Históricos'); tabs.set_title(5, '5. Interpretabilidad'); tabs.set_title(6, '6. DEMO INTERACTIVA')
asset_selector_static = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Activo para Análisis:')

def update_all_tabs(change):
    asset = change['new']
    with tab_children[0]: tab_children[0].clear_output(wait=True); display(create_tab_exploration())
    with tab_children[1]: tab_children[1].clear_output(wait=True); display(create_tab_h1_validation(asset))
    with tab_children[2]: tab_children[2].clear_output(wait=True); display(create_tab_objective1(asset))
    with tab_children[3]: tab_children[3].clear_output(wait=True); display(create_tab_objective2(asset))
    with tab_children[4]: tab_children[4].clear_output(wait=True); display(create_tab_objective3(asset))
    with tab_children[5]: tab_children[5].clear_output(wait=True); display(create_tab_interpretability(asset))

asset_selector_static.observe(update_all_tabs, names='value')

with tab_children[6]:
    tab_children[6].clear_output(wait=True)
    display(create_tab_simulador())

display(HTML("<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las pestañas 0-5. La pestaña 6 (DEMO) tiene su propio selector.</p>")); display(asset_selector_static); display(tabs)
update_all_tabs({'new': 'AMZN'})

Instalando librerías necesarias...
✅ Datos cargados correctamente.
✅ Ingeniería de características completada.
✅ Todos los modelos han sido entrenados y evaluados.


HTML(value='<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las…

Dropdown(description='Activo para Análisis:', options=('AMZN', 'GOOGL'), value='AMZN')

In [ ]:
# ==============================================================================
# PASO 1: INSTALACIÓN Y CARGA DE LIBRERÍAS Y DATOS
# ==============================================================================
print("Instalando librerías necesarias...")
# !pip install pandas scikit-learn statsmodels ipywidgets matplotlib shap -q
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from statsmodels.tsa.arima.model import ARIMA
import ipywidgets as widgets
from ipywidgets import VBox, HBox, HTML, Output, link, Layout
import matplotlib.pyplot as plt
import shap
import warnings

warnings.filterwarnings('ignore')

# Cargar los datos
url = 'https://raw.githubusercontent.com/jaimeisaac2020/Python-analsisis-basicos/mi-github/dataset_completo_google_amazon.csv'
df = pd.read_csv(url, index_col='Date', parse_dates=True)
print("✅ Datos cargados correctamente.")

# ==============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS
# ==============================================================================
df_eng = df.copy()
for asset in ['GOOGL', 'AMZN']:
    df_eng[f'{asset}_Return_1d'] = df_eng[f'{asset}_Close'].pct_change()
    df_eng[f'{asset}_Volatility_20'] = df_eng[f'{asset}_Return_1d'].rolling(window=20).std()
    for lag in range(1, 6): df_eng[f'{asset}_Close_Lag_{lag}'] = df_eng[f'{asset}_Close'].shift(lag)

df_final = df_eng.dropna()
print("✅ Ingeniería de características completada.")

# ==============================================================================
# PASO 3: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS (CÓDIGO CORREGIDO)
# ==============================================================================
results = {}
models = {}
X_train_dict, X_test_dict = {}, {}
features_base_all = ['VIX', 'SP500', 'NASDAQ', 'Treasury_10Y', 'Inflacion_T5YIE']
arima_predictions = {}

for asset in ['GOOGL', 'AMZN']:
    results[asset] = {}
    models[asset] = {}

    train_size = int(len(df_final) * 0.8)
    train_df, test_df = df_final.iloc[:train_size], df_final.iloc[train_size:]
    y_train, y_test = train_df[f'{asset}_Close'], test_df[f'{asset}_Close']

    # --- Experimentos para Pestañas de Objetivos ---
    features_macro_base = ['VIX', 'SP500', 'NASDAQ']
    rf_base = RandomForestRegressor(random_state=42).fit(train_df[features_macro_base], y_train); preds = rf_base.predict(test_df[features_macro_base])
    results[asset]['RF_Base_Macro'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_tasa = features_macro_base + ['Treasury_10Y']
    rf_con_tasa = RandomForestRegressor(random_state=42).fit(train_df[features_con_tasa], y_train); preds = rf_con_tasa.predict(test_df[features_con_tasa])
    results[asset]['RF_Con_Tasa'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_inflacion = features_con_tasa + ['Inflacion_T5YIE']
    rf_con_inf = RandomForestRegressor(random_state=42).fit(train_df[features_con_inflacion], y_train); preds = rf_con_inf.predict(test_df[features_con_inflacion])
    results[asset]['RF_Con_Inflacion'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    # --- Modelos Finales (Enriquecidos) ---
    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]
    features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]

    X_train_enriched, y_train_enriched = train_df[features_enriched], train_df[f'{asset}_Close']
    X_test_enriched, y_test_enriched = test_df[features_enriched], test_df[f'{asset}_Close']
    X_train_dict[asset], X_test_dict[asset] = X_train_enriched, X_test_enriched

    # Entrenar y guardar modelos para el Simulador
    models[asset]['Random Forest'] = RandomForestRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Gradient Boosting'] = GradientBoostingRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Regresión Lineal'] = LinearRegression().fit(X_train_enriched, y_train_enriched)

    # Evaluar y guardar resultados de TODOS los modelos enriquecidos
    for name, model in models[asset].items():
        preds = model.predict(X_test_enriched)
        # CORRECCIÓN: Usar el nombre completo y consistente
        results[asset][f'{name}_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, preds), 'R2': r2_score(y_test_enriched, preds)}

    # Modelo ARIMA
    arima_order = (1, 1, 1) if asset == 'GOOGL' else (0, 1, 0)
    arima_model = ARIMA(y_train_enriched, order=arima_order).fit()
    arima_predictions[asset] = arima_model.forecast(steps=len(y_test_enriched))
    results[asset]['ARIMA_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, arima_predictions[asset]), 'R2': r2_score(y_test_enriched, arima_predictions[asset])}
    models[asset]['ARIMA_pred_estatica'] = arima_predictions[asset].iloc[0]

print("✅ Todos los modelos han sido entrenados y evaluados.")


# ==============================================================================
# PASO 4: FUNCIONES PARA CREAR EL CONTENIDO DE CADA PESTAÑA
# ==============================================================================

# (Las funciones de creación de pestañas y del simulador son idénticas a la versión anterior,
# pero ahora funcionarán porque `results[asset]['Random Forest_Enriquecido']` existe)

def create_tab_exploration():
    head_html = HTML("<h3>Primeros 5 registros (Head)</h3>" + df.head().to_html())
    tail_html = HTML("<h3>Últimos 5 registros (Tail)</h3>" + df.tail().to_html())
    plots_out = Output();
    with plots_out:
        display(HTML("<h4>Gráficos de Series de Tiempo Individuales</h4>"))
        for col in df.columns:
            if col not in ['GOOGL_Return_1d', 'AMZN_Return_1d', 'GOOGL_Volatility_20', 'AMZN_Volatility_20']:
                fig, ax = plt.subplots(figsize=(8, 2)); df[col].plot(ax=ax, title=col); plt.tight_layout(); plt.show()
    return VBox([head_html, tail_html, plots_out])

def create_tab_h1_validation(asset):
    asset_results = {
        'ARIMA': results[asset]['ARIMA_Enriquecido'],
        'Random Forest': results[asset]['Random Forest_Enriquecido'],
        'Gradient Boosting': results[asset]['Gradient Boosting_Enriquecido']
    }
    df_res = pd.DataFrame(asset_results).T
    reduction_vs_arima = (1 - df_res.loc['Random Forest', 'MSE'] / df_res.loc['ARIMA', 'MSE']) * 100 if df_res.loc['ARIMA', 'MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())

    graph_out = Output()
    with graph_out:
        fig, ax = plt.subplots(figsize=(10, 5))
        test_dates = test_df.index
        ax.plot(test_dates, test_df[f'{asset}_Close'], label='Precio Real', color='black', linewidth=2)
        ax.plot(test_dates, arima_predictions[asset], label='ARIMA (Tradicional)', color='red', linestyle='--')
        ax.plot(test_dates, models[asset]['Random Forest'].predict(X_test_dict[asset]), label='Random Forest (Supervisado)', color='blue', linestyle='--')
        ax.set_title(f'Comparación Visual de Predicciones para {asset}'); ax.legend(); plt.tight_layout(); plt.show()

    return VBox([HTML("<h3>Validación Hipótesis 1: Superioridad de Modelos Supervisados</h3>"), HTML("<p>Ligada al <b>Objetivo Específico 4</b>.</p>"), HTML("<hr><h4>Evidencia Cuantitativa</h4>"), styled_table_html, HTML(f"<div style='margin-top:10px; padding:10px; background-color:#d4edda; border-radius:5px;'><b>Conclusión:</b> Random Forest reduce el error (MSE) en un <b>{reduction_vs_arima:.2f}%</b> vs. ARIMA para {asset}.</div>"), HTML("<hr><h4>Evidencia Visual</h4>"), graph_out, HTML("<p><b>Interpretación:</b> ARIMA (rojo) no captura la tendencia, mientras Random Forest (azul) se adapta a la dinámica real.</p>"), HTML("<hr><h3 style='color:green;'>Veredicto: Hipótesis 1 Aceptada</h3>")])

def create_tab_objective1(asset):
    res_base = results[asset]['RF_Base_Macro']; res_completo = results[asset]['RF_Con_Tasa']
    df_res = pd.DataFrame([res_base, res_completo], index=['Modelo Base', 'Modelo con Tasa Int.'])
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue'); ax1.set_ylabel(f'Precio {asset}', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Treasury_10Y'], color='red', linestyle='--'); ax2.set_ylabel('Tasa 10Y (%)', color='red'); ax2.tick_params(axis='y', labelcolor='red'); plt.title(f'Contexto: Precio vs. Tasa de Interés'); plt.show()
    results_plot_out = Output()
    with results_plot_out:
        fig, ax = plt.subplots(figsize=(5, 3)); bars = ax.bar(df_res.index, df_res['MSE'], color=['gray', 'skyblue']); ax.set_title(f'Reducción de Error'); ax.bar_label(bars, fmt='{:,.2f}'); plt.show()
    return VBox([HTML("<h4>Validación H2: Impacto de Tasas de Interés</h4>"), context_plot_out, HBox([styled_table_html, results_plot_out])])

def create_tab_objective2(asset):
    res_sin = results[asset]['RF_Con_Tasa']; res_con = results[asset]['RF_Con_Inflacion']
    df_res = pd.DataFrame([res_sin, res_con], index=['Modelo SIN Inflación', 'Modelo CON Inflación'])
    incremento_error = (res_sin['MSE'] / res_con['MSE'] - 1) * 100 if res_con['MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue'); ax1.set_ylabel(f'Precio {asset}', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Inflacion_T5YIE'], color='green', linestyle='--'); ax2.set_ylabel('Inflación 5A (%)', color='green'); ax2.tick_params(axis='y', labelcolor='green'); plt.title(f'Contexto: Precio vs. Inflación'); plt.show()
    return VBox([HTML("<h4>Validación H3: Impacto de la Inflación</h4>"), context_plot_out, styled_table_html, HTML(f"<div style='margin-top:10px; padding:10px; background-color:#f8d7da; border-radius:5px;'>Eliminar la inflación <b>aumentó el error en un {incremento_error:.2f}%</b> para {asset}.</div>")])

def create_tab_objective3(asset):
    res_base = results[asset]['RF_Base_Macro']
    # CORRECCIÓN: Llamar el resultado guardado con el nombre correcto
    res_enriquecido = results[asset]['Random Forest_Enriquecido']
    df_res = pd.DataFrame([res_base, res_enriquecido], index=['Modelo SIN Históricos', 'Modelo CON Históricos'])
    r2_color = 'color:red' if df_res.loc['Modelo CON Históricos', 'R2'] < 0 else 'color:green'
    html_resumen = (f"<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>¡Resultado Clave para {asset}!</b> El R² pasó de {res_base['R2']:.2f} a <b style='{r2_color}'>{res_enriquecido['R2']:.2f}</b>.</div>") if asset == 'AMZN' else ""
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())
    return VBox([HTML("<h4>Validación H4: Impacto de Datos Históricos</h4>"), HTML(html_resumen), styled_table_html])

def create_tab_interpretability(asset):
    # (código sin cambios)
    model = models[asset]['Random Forest']; X_test = X_test_dict[asset]; X_train = X_train_dict[asset]
    shap_out = Output()
    with shap_out:
        explainer = shap.TreeExplainer(model); shap_values = explainer(X_test); plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_values, X_test, show=False, plot_size=None); plt.title(f'SHAP Summary Plot para {asset}'); plt.tight_layout(); plt.show()
    pdp_out = Output()
    with pdp_out:
        features_to_plot = ['Treasury_10Y', 'Inflacion_T5YIE', 'VIX', f'{asset}_Close_Lag_1']
        fig, ax = plt.subplots(1, len(features_to_plot), figsize=(15, 3))
        PartialDependenceDisplay.from_estimator(model, X_train, features_to_plot, ax=ax); plt.suptitle(f'Partial Dependence Plots (PDP) para {asset}'); plt.tight_layout(); plt.show()
    return VBox([HTML("<h3>Interpretabilidad del Modelo Random Forest</h3>"), HTML("<h4>Gráfico SHAP</h4>"), shap_out, HTML("<hr><h4>Gráficos PDP</h4>"), pdp_out])

def create_tab_simulador():
    # (código sin cambios)
    asset_selector_sim = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Seleccione el Activo:')
    tasa_interes_slider = widgets.FloatSlider(value=4.0, min=2.0, max=6.0, step=0.05, description='Tasa Interés:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    inflacion_slider = widgets.FloatSlider(value=2.5, min=1.5, max=4.0, step=0.05, description='Inflación:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    vix_slider = widgets.FloatSlider(value=15, min=10, max=40, step=1, description='VIX:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    precio_anterior_slider = widgets.FloatSlider(value=df_final['AMZN_Close'].iloc[-1], min=150, max=250, step=0.5, description='Precio Anterior:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    tasa_interes_text, inflacion_text, vix_text, precio_anterior_text = (widgets.FloatText(layout=Layout(width='20%')) for _ in range(4))
    link((tasa_interes_slider, 'value'), (tasa_interes_text, 'value')); link((inflacion_slider, 'value'), (inflacion_text, 'value')); link((vix_slider, 'value'), (vix_text, 'value')); link((precio_anterior_slider, 'value'), (precio_anterior_text, 'value'))
    table_output_container = HTML(); graph_output_container = Output()

    def update_simulador(asset, tasa_interes, inflacion, vix, precio_anterior):
        if any(v is None for v in [tasa_interes, inflacion, vix, precio_anterior]): return
        last_known = df_final.iloc[-1].copy(); input_data = pd.DataFrame([last_known])
        input_data['Treasury_10Y'], input_data['Inflacion_T5YIE'], input_data['VIX'], input_data[f'{asset}_Close_Lag_1'] = tasa_interes, inflacion, vix, precio_anterior
        for lag in range(2, 6): input_data[f'{asset}_Close_Lag_{lag}'] = precio_anterior
        features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
        input_data_final = input_data[features_enriched]
        preds = {name: model.predict(input_data_final)[0] for name, model in models[asset].items() if name != 'ARIMA_pred_estatica'}; preds['ARIMA'] = models[asset]['ARIMA_pred_estatica']
        table_output_container.value = create_html_table(preds, precio_anterior)
        graph_output_container.clear_output(wait=True)
        with graph_output_container: create_prediction_graph(df_final[f'{asset}_Close'], preds, asset, precio_anterior)

    def create_html_table(preds, precio_anterior):
        best_model_name = "Random Forest"
        html = f"""<div style="border:1px solid #ccc;padding:15px;border-radius:8px;"><h3 style="margin-top:0;">Tabla de Predicciones</h3><p style="font-size:0.9em;">Precio Anterior: <b>{precio_anterior:,.2f} USD</b></p><table style="width:100%; border-collapse: collapse;"><thead><tr style="background-color:#005a9c;color:white;"><th style="padding:10px;text-align:left;">Modelo</th><th style="padding:10px;text-align:right;">Precio Predicho</th></tr></thead><tbody>"""
        for model_name in ['Random Forest', 'Gradient Boosting', 'Regresión Lineal', 'ARIMA']:
            pred = preds[model_name]; is_best = (model_name == best_model_name); style = 'background-color:#d4edda;color:#155724;font-weight:bold;' if is_best else ''
            tag = ' <span style="font-size:0.8em;background-color:#28a745;color:white;padding:2px 5px;border-radius:3px;">Mejor Modelo</span>' if is_best else ''
            static_tag = ' <span style="font-size:0.8em;background-color:#dc3545;color:white;padding:2px 5px;border-radius:3px;">Estático</span>' if model_name == 'ARIMA' else ''
            html += f"""<tr style="{style}"><td style="padding:8px;border-bottom:1px solid #ddd;">{model_name}{tag}{static_tag}</td><td style="padding:8px;border-bottom:1px solid #ddd;text-align:right;">{pred:,.2f}</td></tr>"""
        return html + "</tbody></table></div>"

    def create_prediction_graph(historic_data_series, preds, asset, precio_anterior):
        plt.style.use('seaborn-v0_8-whitegrid'); fig, ax = plt.subplots(figsize=(9, 5)); historic_data = historic_data_series.tail(100)
        ax.plot(historic_data.index, historic_data.values, label=f'Histórico de {asset}', color='black', alpha=0.7)
        last_date = historic_data.index[-1]; prediction_date = last_date + pd.Timedelta(days=1); colors = {'ARIMA': 'red', 'Regresión Lineal': 'orange', 'Gradient Boosting': 'purple', 'Random Forest': 'blue'}; best_model_name = "Random Forest"
        for model_name, prediction in preds.items():
            is_best = (model_name == best_model_name)
            ax.plot([last_date, prediction_date], [precio_anterior, prediction], linestyle='--', marker='o', color=colors[model_name], label=f'{model_name}: {prediction:,.2f}', linewidth=3.0 if is_best else 1.5, alpha=1.0 if is_best else 0.8)
        ax.set_title(f'Pronósticos para {asset} bajo Escenario Simulado', fontsize=16); ax.set_ylabel('Precio (USD)'); ax.legend(loc='best'); plt.tight_layout(); plt.show()

    def on_asset_change_simulador(change):
        asset = change['new']; last_price = df_final[f'{asset}_Close'].iloc[-1]
        precio_anterior_slider.min = last_price * 0.8; precio_anterior_slider.max = last_price * 1.2
        precio_anterior_slider.value = last_price

    asset_selector_sim.observe(on_asset_change_simulador, names='value')
    controles_sliders = VBox([HBox([tasa_interes_slider, tasa_interes_text]), HBox([inflacion_slider, inflacion_text]), HBox([vix_slider, vix_text]), HBox([precio_anterior_slider, precio_anterior_text])])
    controles_principales = VBox([asset_selector_sim, HTML("<hr><p><b>Defina un escenario:</b></p>"), controles_sliders])
    resultados_ui = HBox([table_output_container, graph_output_container], layout=Layout(align_items='flex-start', justify_content='space-between'))
    widgets.interactive_output(update_simulador, {'asset': asset_selector_sim, 'tasa_interes': tasa_interes_slider, 'inflacion': inflacion_slider, 'vix': vix_slider, 'precio_anterior': precio_anterior_slider})
    on_asset_change_simulador({'new': asset_selector_sim.value})
    return VBox([HTML("<h3>Simulador de Escenarios en Vivo</h3>"), controles_principales, resultados_ui])

# --- Montaje Final del Dashboard ---
tab_children = [Output() for _ in range(7)]
tabs = widgets.Tab(children=tab_children)
tabs.set_title(0, '0. Exploración Datos'); tabs.set_title(1, '1. Validación H1'); tabs.set_title(2, '2. Impacto Tasas Int.'); tabs.set_title(3, '3. Impacto Inflación'); tabs.set_title(4, '4. Impacto D. Históricos'); tabs.set_title(5, '5. Interpretabilidad'); tabs.set_title(6, '6. DEMO INTERACTIVA')
asset_selector_static = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Activo para Análisis:')

def update_all_tabs(change):
    asset = change['new']
    with tab_children[0]: tab_children[0].clear_output(wait=True); display(create_tab_exploration())
    with tab_children[1]: tab_children[1].clear_output(wait=True); display(create_tab_h1_validation(asset))
    with tab_children[2]: tab_children[2].clear_output(wait=True); display(create_tab_objective1(asset))
    with tab_children[3]: tab_children[3].clear_output(wait=True); display(create_tab_objective2(asset))
    with tab_children[4]: tab_children[4].clear_output(wait=True); display(create_tab_objective3(asset))
    with tab_children[5]: tab_children[5].clear_output(wait=True); display(create_tab_interpretability(asset))

asset_selector_static.observe(update_all_tabs, names='value')

with tab_children[6]:
    tab_children[6].clear_output(wait=True)
    display(create_tab_simulador())

display(HTML("<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las pestañas 0-5. La pestaña 6 (DEMO) tiene su propio selector.</p>")); display(asset_selector_static); display(tabs)
update_all_tabs({'new': 'AMZN'})

Instalando librerías necesarias...
✅ Datos cargados correctamente.
✅ Ingeniería de características completada.
✅ Todos los modelos han sido entrenados y evaluados.


HTML(value='<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las…

Dropdown(description='Activo para Análisis:', options=('AMZN', 'GOOGL'), value='AMZN')

In [ ]:
# ==============================================================================
# PASO 1: INSTALACIÓN Y CARGA DE LIBRERÍAS Y DATOS
# ==============================================================================
print("Instalando librerías necesarias... (puede tardar un momento)")
# !pip install pandas scikit-learn statsmodels ipywidgets matplotlib shap -q
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from statsmodels.tsa.arima.model import ARIMA
import ipywidgets as widgets
from ipywidgets import VBox, HBox, HTML, Output, link, Layout
import matplotlib.pyplot as plt
import shap
import warnings

warnings.filterwarnings('ignore')

# Cargar los datos
url = 'https://raw.githubusercontent.com/jaimeisaac2020/Python-analsisis-basicos/mi-github/dataset_completo_google_amazon.csv'
df = pd.read_csv(url, index_col='Date', parse_dates=True)
print("✅ Datos cargados correctamente.")

# ==============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS
# ==============================================================================
df_eng = df.copy()
for asset in ['GOOGL', 'AMZN']:
    df_eng[f'{asset}_Return_1d'] = df_eng[f'{asset}_Close'].pct_change()
    df_eng[f'{asset}_Volatility_20'] = df_eng[f'{asset}_Return_1d'].rolling(window=20).std()
    for lag in range(1, 6): df_eng[f'{asset}_Close_Lag_{lag}'] = df_eng[f'{asset}_Close'].shift(lag)

df_final = df_eng.dropna()
print("✅ Ingeniería de características completada.")

# ==============================================================================
# PASO 3: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS
# ==============================================================================
results = {}
models = {}
X_train_dict, X_test_dict = {}, {}
features_base_all = ['VIX', 'SP500', 'NASDAQ', 'Treasury_10Y', 'Inflacion_T5YIE']
arima_predictions = {}

for asset in ['GOOGL', 'AMZN']:
    results[asset] = {}
    models[asset] = {}

    train_size = int(len(df_final) * 0.8)
    train_df, test_df = df_final.iloc[:train_size], df_final.iloc[train_size:]
    y_train, y_test = train_df[f'{asset}_Close'], test_df[f'{asset}_Close']

    features_macro_base = ['VIX', 'SP500', 'NASDAQ']
    rf_base = RandomForestRegressor(random_state=42).fit(train_df[features_macro_base], y_train); preds = rf_base.predict(test_df[features_macro_base])
    results[asset]['RF_Base_Macro'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_tasa = features_macro_base + ['Treasury_10Y']
    rf_con_tasa = RandomForestRegressor(random_state=42).fit(train_df[features_con_tasa], y_train); preds = rf_con_tasa.predict(test_df[features_con_tasa])
    results[asset]['RF_Con_Tasa'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_inflacion = features_con_tasa + ['Inflacion_T5YIE']
    rf_con_inf = RandomForestRegressor(random_state=42).fit(train_df[features_con_inflacion], y_train); preds = rf_con_inf.predict(test_df[features_con_inflacion])
    results[asset]['RF_Con_Inflacion'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
    X_train_enriched, y_train_enriched = train_df[features_enriched], train_df[f'{asset}_Close']
    X_test_enriched, y_test_enriched = test_df[features_enriched], test_df[f'{asset}_Close']
    X_train_dict[asset], X_test_dict[asset] = X_train_enriched, X_test_enriched

    models[asset]['Random Forest'] = RandomForestRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Gradient Boosting'] = GradientBoostingRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Regresión Lineal'] = LinearRegression().fit(X_train_enriched, y_train_enriched)

    for name, model in models[asset].items():
        preds = model.predict(X_test_enriched)
        results[asset][f'{name}_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, preds), 'R2': r2_score(y_test_enriched, preds)}

    arima_order = (1, 1, 1) if asset == 'GOOGL' else (0, 1, 0)
    arima_model = ARIMA(y_train_enriched, order=arima_order).fit()
    arima_predictions[asset] = arima_model.forecast(steps=len(y_test_enriched))
    results[asset]['ARIMA_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, arima_predictions[asset]), 'R2': r2_score(y_test_enriched, arima_predictions[asset])}
    models[asset]['ARIMA_pred_estatica'] = arima_predictions[asset].iloc[0]

print("✅ Todos los modelos han sido entrenados y evaluados.")

# ==============================================================================
# PASO 4: FUNCIONES PARA CREAR EL CONTENIDO DE CADA PESTAÑA
# ==============================================================================
def create_tab_exploration():
    head_html = HTML("<h3>Primeros 5 registros (Head)</h3>" + df.head().to_html())
    tail_html = HTML("<h3>Últimos 5 registros (Tail)</h3>" + df.tail().to_html())
    plots_out = Output()
    with plots_out:
        display(HTML("<h4>Gráficos de Series de Tiempo Individuales</h4>"))
        for col in df.columns:
            if 'Lag' not in col and 'Return' not in col and 'Volatility' not in col:
                fig, ax = plt.subplots(figsize=(8, 2)); df[col].plot(ax=ax, title=col); plt.tight_layout(); plt.show()
    return VBox([head_html, tail_html, plots_out])

def create_tab_h1_validation(asset):
    asset_results = {'ARIMA': results[asset]['ARIMA_Enriquecido'], 'Random Forest': results[asset]['Random Forest_Enriquecido'], 'Gradient Boosting': results[asset]['Gradient Boosting_Enriquecido']}
    df_res = pd.DataFrame(asset_results).T
    reduction_vs_arima = (1 - df_res.loc['Random Forest', 'MSE'] / df_res.loc['ARIMA', 'MSE']) * 100 if df_res.loc['ARIMA', 'MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())
    graph_out = Output()
    with graph_out:
        fig, ax = plt.subplots(figsize=(10, 5)); test_dates = test_df.index
        ax.plot(test_dates, test_df[f'{asset}_Close'], label='Precio Real', color='black', linewidth=2)
        ax.plot(test_dates, arima_predictions[asset], label='ARIMA (Tradicional)', color='red', linestyle='--')
        ax.plot(test_dates, models[asset]['Random Forest'].predict(X_test_dict[asset]), label='Random Forest (Supervisado)', color='blue', linestyle='--')
        ax.set_title(f'Comparación Visual de Predicciones para {asset}'); ax.legend(); plt.tight_layout(); plt.show()
    return VBox([HTML("<h3>Validación Hipótesis 1: Superioridad de Modelos Supervisados</h3><p>Ligada al <b>Objetivo Específico 4</b>.</p>"), HTML("<hr><h4>Evidencia Cuantitativa</h4>"), styled_table_html, HTML(f"<div style='margin-top:10px; padding:10px; background-color:#d4edda; border-radius:5px;'><b>Conclusión Cuantitativa:</b> Random Forest reduce el error (MSE) en un <b>{reduction_vs_arima:.2f}%</b> vs. ARIMA para {asset}.</div>"), HTML("<hr><h4>Evidencia Visual</h4>"), graph_out, HTML("<p><b>Interpretación Visual:</b> ARIMA (rojo) no captura la tendencia, mientras Random Forest (azul) se adapta a la dinámica real.</p>"), HTML("<hr><h3 style='color:green;'>Veredicto: Hipótesis 1 Aceptada</h3>")])

def create_tab_objective1(asset):
    res_base = results[asset]['RF_Base_Macro']; res_completo = results[asset]['RF_Con_Tasa']
    df_res = pd.DataFrame([res_base, res_completo], index=['Modelo Base', 'Modelo con Tasa Int.']); styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output();
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue'); ax1.set_ylabel(f'Precio {asset}', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Treasury_10Y'], color='red', linestyle='--'); ax2.set_ylabel('Tasa 10Y (%)', color='red'); ax2.tick_params(axis='y', labelcolor='red'); plt.title(f'Contexto: Precio vs. Tasa de Interés'); plt.show()
    results_plot_out = Output()
    with results_plot_out:
        fig, ax = plt.subplots(figsize=(5, 3)); bars = ax.bar(df_res.index, df_res['MSE'], color=['gray', 'skyblue']); ax.set_title(f'Reducción de Error'); ax.bar_label(bars, fmt='{:,.2f}'); plt.show()
    return VBox([HTML("<h4>Validación H2: Impacto de Tasas de Interés</h4>"), context_plot_out, HBox([styled_table_html, results_plot_out])])

def create_tab_objective2(asset):
    res_sin = results[asset]['RF_Con_Tasa']; res_con = results[asset]['RF_Con_Inflacion']
    df_res = pd.DataFrame([res_sin, res_con], index=['Modelo SIN Inflación', 'Modelo CON Inflación'])
    incremento_error = (res_sin['MSE'] / res_con['MSE'] - 1) * 100 if res_con['MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue'); ax1.set_ylabel(f'Precio {asset}', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Inflacion_T5YIE'], color='green', linestyle='--'); ax2.set_ylabel('Inflación 5A (%)', color='green'); ax2.tick_params(axis='y', labelcolor='green'); plt.title(f'Contexto: Precio vs. Inflación'); plt.show()
    return VBox([HTML("<h4>Validación H3: Impacto de la Inflación</h4>"), context_plot_out, styled_table_html, HTML(f"<div style='margin-top:10px; padding:10px; background-color:#f8d7da; border-radius:5px;'>Eliminar la inflación <b>aumentó el error en un {incremento_error:.2f}%</b> para {asset}.</div>")])

def create_tab_objective3(asset):
    res_base = results[asset]['RF_Base_Macro']; res_enriquecido = results[asset]['Random Forest_Enriquecido']
    df_res = pd.DataFrame([res_base, res_enriquecido], index=['Modelo SIN Históricos', 'Modelo CON Históricos'])
    r2_color = 'color:red' if df_res.loc['Modelo CON Históricos', 'R2'] < 0 else 'color:green'
    html_resumen = (f"<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>¡Resultado Clave para {asset}!</b> El R² pasó de {res_base['R2']:.2f} a <b style='{r2_color}'>{res_enriquecido['R2']:.2f}</b>.</div>") if asset == 'AMZN' else ""
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())
    return VBox([HTML("<h4>Validación H4: Impacto de Datos Históricos</h4>"), HTML(html_resumen), styled_table_html])

def create_tab_interpretability(asset):
    model = models[asset]['Random Forest']; X_test = X_test_dict[asset]; X_train = X_train_dict[asset]
    shap_out = Output()
    with shap_out:
        explainer = shap.TreeExplainer(model); shap_values = explainer(X_test); plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_values, X_test, show=False, plot_size=None); plt.title(f'SHAP Summary Plot para {asset}'); plt.tight_layout(); plt.show()
    pdp_out = Output()
    with pdp_out:
        features_to_plot = ['Treasury_10Y', 'Inflacion_T5YIE', 'VIX', f'{asset}_Close_Lag_1']
        fig, ax = plt.subplots(1, len(features_to_plot), figsize=(15, 3))
        PartialDependenceDisplay.from_estimator(model, X_train, features_to_plot, ax=ax); plt.suptitle(f'Partial Dependence Plots (PDP) para {asset}'); plt.tight_layout(); plt.show()
    return VBox([HTML("<h3>Interpretabilidad del Modelo Random Forest</h3>"), HTML("<h4>Gráfico SHAP: Importancia y Dirección de Variables</h4>"), shap_out, HTML("<hr><h4>Gráficos PDP: Relación Promedio de Variables Clave</h4>"), pdp_out])

def create_tab_simulador():
    asset_selector_sim = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Seleccione el Activo:')
    tasa_interes_slider = widgets.FloatSlider(value=4.0, min=2.0, max=6.0, step=0.05, description='Tasa Interés:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    inflacion_slider = widgets.FloatSlider(value=2.5, min=1.5, max=4.0, step=0.05, description='Inflación:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    vix_slider = widgets.FloatSlider(value=15, min=10, max=40, step=1, description='VIX:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    precio_anterior_slider = widgets.FloatSlider(value=df_final['AMZN_Close'].iloc[-1], min=150, max=250, step=0.5, description='Precio Anterior:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    tasa_interes_text, inflacion_text, vix_text, precio_anterior_text = (widgets.FloatText(layout=Layout(width='20%')) for _ in range(4))
    link((tasa_interes_slider, 'value'), (tasa_interes_text, 'value')); link((inflacion_slider, 'value'), (inflacion_text, 'value')); link((vix_slider, 'value'), (vix_text, 'value')); link((precio_anterior_slider, 'value'), (precio_anterior_text, 'value'))
    table_output_container = HTML(); graph_output_container = Output()

    def update_simulador(asset, tasa_interes, inflacion, vix, precio_anterior):
        if any(v is None for v in [tasa_interes, inflacion, vix, precio_anterior]): return
        last_known = df_final.iloc[-1].copy(); input_data = pd.DataFrame([last_known])
        input_data['Treasury_10Y'], input_data['Inflacion_T5YIE'], input_data['VIX'], input_data[f'{asset}_Close_Lag_1'] = tasa_interes, inflacion, vix, precio_anterior
        for lag in range(2, 6): input_data[f'{asset}_Close_Lag_{lag}'] = precio_anterior
        features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
        input_data_final = input_data[features_enriched]
        preds = {name: model.predict(input_data_final)[0] for name, model in models[asset].items() if name != 'ARIMA_pred_estatica'}; preds['ARIMA'] = models[asset]['ARIMA_pred_estatica']
        table_output_container.value = create_html_table(preds, precio_anterior)
        graph_output_container.clear_output(wait=True)
        with graph_output_container: create_prediction_graph(df_final[f'{asset}_Close'], preds, asset, precio_anterior)

    def create_html_table(preds, precio_anterior):
        best_model_name = "Random Forest"
        html = f"""<div style="border:1px solid #ccc;padding:15px;border-radius:8px;"><h3 style="margin-top:0;">Tabla de Predicciones</h3><p style="font-size:0.9em;">Precio Anterior: <b>{precio_anterior:,.2f} USD</b></p><table style="width:100%; border-collapse: collapse;"><thead><tr style="background-color:#005a9c;color:white;"><th style="padding:10px;text-align:left;">Modelo</th><th style="padding:10px;text-align:right;">Precio Predicho</th></tr></thead><tbody>"""
        for model_name in ['Random Forest', 'Gradient Boosting', 'Regresión Lineal', 'ARIMA']:
            pred = preds[model_name]; is_best = (model_name == best_model_name); style = 'background-color:#d4edda;color:#155724;font-weight:bold;' if is_best else ''
            tag = ' <span style="font-size:0.8em;background-color:#28a745;color:white;padding:2px 5px;border-radius:3px;">Mejor Modelo</span>' if is_best else ''
            static_tag = ' <span style="font-size:0.8em;background-color:#dc3545;color:white;padding:2px 5px;border-radius:3px;">Estático</span>' if model_name == 'ARIMA' else ''
            html += f"""<tr style="{style}"><td style="padding:8px;border-bottom:1px solid #ddd;">{model_name}{tag}{static_tag}</td><td style="padding:8px;border-bottom:1px solid #ddd;text-align:right;">{pred:,.2f}</td></tr>"""
        return html + "</tbody></table></div>"

    def create_prediction_graph(historic_data_series, preds, asset, precio_anterior):
        plt.style.use('seaborn-v0_8-whitegrid'); fig, ax = plt.subplots(figsize=(9, 5)); historic_data = historic_data_series.tail(100)
        ax.plot(historic_data.index, historic_data.values, label=f'Histórico de {asset}', color='black', alpha=0.7)
        last_date = historic_data.index[-1]; prediction_date = last_date + pd.Timedelta(days=1); colors = {'ARIMA': 'red', 'Regresión Lineal': 'orange', 'Gradient Boosting': 'purple', 'Random Forest': 'blue'}; best_model_name = "Random Forest"
        for model_name, prediction in preds.items():
            is_best = (model_name == best_model_name)
            ax.plot([last_date, prediction_date], [precio_anterior, prediction], linestyle='--', marker='o', color=colors[model_name], label=f'{model_name}: {prediction:,.2f}', linewidth=3.0 if is_best else 1.5, alpha=1.0 if is_best else 0.8)
        ax.set_title(f'Pronósticos para {asset} bajo Escenario Simulado', fontsize=16); ax.set_ylabel('Precio (USD)'); ax.legend(loc='best'); plt.tight_layout(); plt.show()

    def on_asset_change_simulador(change):
        asset = change['new']; last_price = df_final[f'{asset}_Close'].iloc[-1]
        precio_anterior_slider.min = last_price * 0.8; precio_anterior_slider.max = last_price * 1.2
        precio_anterior_slider.value = last_price

    asset_selector_sim.observe(on_asset_change_simulador, names='value')
    controles_sliders = VBox([HBox([tasa_interes_slider, tasa_interes_text]), HBox([inflacion_slider, inflacion_text]), HBox([vix_slider, vix_text]), HBox([precio_anterior_slider, precio_anterior_text])])
    controles_principales = VBox([asset_selector_sim, HTML("<hr><p><b>Defina un escenario:</b></p>"), controles_sliders])
    resultados_ui = HBox([table_output_container, graph_output_container], layout=Layout(align_items='flex-start', justify_content='space-between'))
    widgets.interactive_output(update_simulador, {'asset': asset_selector_sim, 'tasa_interes': tasa_interes_slider, 'inflacion': inflacion_slider, 'vix': vix_slider, 'precio_anterior': precio_anterior_slider})
    on_asset_change_simulador({'new': asset_selector_sim.value})
    return VBox([HTML("<h3>Simulador de Escenarios en Vivo</h3>"), controles_principales, resultados_ui])

# --- Montaje Final del Dashboard ---
tab_children = [Output() for _ in range(7)]
tabs = widgets.Tab(children=tab_children)
tabs.set_title(0, '0. Exploración Datos'); tabs.set_title(1, '1. Validación H1'); tabs.set_title(2, '2. Impacto Tasas Int.'); tabs.set_title(3, '3. Impacto Inflación'); tabs.set_title(4, '4. Impacto D. Históricos'); tabs.set_title(5, '5. Interpretabilidad'); tabs.set_title(6, '6. DEMO INTERACTIVA')
asset_selector_static = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Activo para Análisis:')

def update_all_tabs(change):
    asset = change['new']
    with tab_children[0]: tab_children[0].clear_output(wait=True); display(create_tab_exploration())
    with tab_children[1]: tab_children[1].clear_output(wait=True); display(create_tab_h1_validation(asset))
    with tab_children[2]: tab_children[2].clear_output(wait=True); display(create_tab_objective1(asset))
    with tab_children[3]: tab_children[3].clear_output(wait=True); display(create_tab_objective2(asset))
    with tab_children[4]: tab_children[4].clear_output(wait=True); display(create_tab_objective3(asset))
    with tab_children[5]: tab_children[5].clear_output(wait=True); display(create_tab_interpretability(asset))

asset_selector_static.observe(update_all_tabs, names='value')

with tab_children[6]:
    tab_children[6].clear_output(wait=True)
    display(create_tab_simulador())

display(HTML("<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las pestañas 0-5. La pestaña 6 (DEMO) tiene su propio selector.</p>")); display(asset_selector_static); display(tabs)
update_all_tabs({'new': 'AMZN'})

Instalando librerías necesarias... (puede tardar un momento)
✅ Datos cargados correctamente.
✅ Ingeniería de características completada.
✅ Todos los modelos han sido entrenados y evaluados.


HTML(value='<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las…

Dropdown(description='Activo para Análisis:', options=('AMZN', 'GOOGL'), value='AMZN')

In [4]:
# ==============================================================================
# PASO 1: INSTALACIÓN Y CARGA DE LIBRERÍAS Y DATOS
# ==============================================================================
print("Instalando librerías necesarias... (puede tardar un momento la primera vez)")
try:
    import shap
except ImportError:
    # !pip install pandas scikit-learn statsmodels ipywidgets matplotlib shap -q
    print("Instalación completada.")

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
from statsmodels.tsa.arima.model import ARIMA
import ipywidgets as widgets
from ipywidgets import VBox, HBox, HTML, Output, link, Layout
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Cargar los datos
url = 'https://raw.githubusercontent.com/jaimeisaac2020/Python-analsisis-basicos/mi-github/dataset_completo_google_amazon.csv'
df = pd.read_csv(url, index_col='Date', parse_dates=True)
print("✅ Datos cargados correctamente.")

# ==============================================================================
# PASO 2: INGENIERÍA DE CARACTERÍSTICAS
# ==============================================================================
df_eng = df.copy()
for asset in ['GOOGL', 'AMZN']:
    df_eng[f'{asset}_Return_1d'] = df_eng[f'{asset}_Close'].pct_change()
    df_eng[f'{asset}_Volatility_20'] = df_eng[f'{asset}_Return_1d'].rolling(window=20).std()
    for lag in range(1, 6): df_eng[f'{asset}_Close_Lag_{lag}'] = df_eng[f'{asset}_Close'].shift(lag)

df_final = df_eng.dropna()
print("✅ Ingeniería de características completada.")

# ==============================================================================
# PASO 3: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS
# ==============================================================================
results = {}
models = {}
X_train_dict = {}
X_test_dict = {}
features_base_all = ['VIX', 'SP500', 'NASDAQ', 'Treasury_10Y', 'Inflacion_T5YIE']
arima_predictions = {}

for asset in ['GOOGL', 'AMZN']:
    results[asset] = {}
    models[asset] = {}

    train_size = int(len(df_final) * 0.8)
    train_df, test_df = df_final.iloc[:train_size], df_final.iloc[train_size:]
    y_train, y_test = train_df[f'{asset}_Close'], test_df[f'{asset}_Close']

    features_macro_base = ['VIX', 'SP500', 'NASDAQ']
    rf_base = RandomForestRegressor(random_state=42).fit(train_df[features_macro_base], y_train); preds = rf_base.predict(test_df[features_macro_base])
    results[asset]['RF_Base_Macro'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_tasa = features_macro_base + ['Treasury_10Y']
    rf_con_tasa = RandomForestRegressor(random_state=42).fit(train_df[features_con_tasa], y_train); preds = rf_con_tasa.predict(test_df[features_con_tasa])
    results[asset]['RF_Con_Tasa'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_con_inflacion = features_con_tasa + ['Inflacion_T5YIE']
    rf_con_inf = RandomForestRegressor(random_state=42).fit(train_df[features_con_inflacion], y_train); preds = rf_con_inf.predict(test_df[features_con_inflacion])
    results[asset]['RF_Con_Inflacion'] = {'MSE': mean_squared_error(y_test, preds), 'R2': r2_score(y_test, preds)}

    features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
    X_train_enriched, y_train_enriched = train_df[features_enriched], train_df[f'{asset}_Close']
    X_test_enriched, y_test_enriched = test_df[features_enriched], test_df[f'{asset}_Close']
    X_train_dict[asset], X_test_dict[asset] = X_train_enriched, X_test_enriched

    models[asset]['Random Forest'] = RandomForestRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Gradient Boosting'] = GradientBoostingRegressor(random_state=42).fit(X_train_enriched, y_train_enriched)
    models[asset]['Regresión Lineal'] = LinearRegression().fit(X_train_enriched, y_train_enriched)

    for name, model in models[asset].items():
        preds = model.predict(X_test_enriched)
        results[asset][f'{name}_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, preds), 'R2': r2_score(y_test_enriched, preds)}

    arima_order = (1, 1, 1) if asset == 'GOOGL' else (0, 1, 0)
    arima_model = ARIMA(y_train_enriched, order=arima_order).fit()
    arima_predictions[asset] = arima_model.forecast(steps=len(y_test_enriched))
    results[asset]['ARIMA_Enriquecido'] = {'MSE': mean_squared_error(y_test_enriched, arima_predictions[asset]), 'R2': r2_score(y_test_enriched, arima_predictions[asset])}
    models[asset]['ARIMA_pred_estatica'] = arima_predictions[asset].iloc[0]

print("✅ Todos los modelos han sido entrenados y evaluados.")

# ==============================================================================
# PASO 4: FUNCIONES PARA CREAR EL CONTENIDO DE LAS PESTAÑAS
# ==============================================================================

# --- (Las funciones para las pestañas 0, 1, 2, 3, 4 y 5 no necesitan cambios) ---
def create_tab_exploration():
    head_html = HTML("<h3>Primeros 5 registros (Head)</h3>" + df.head().to_html())
    tail_html = HTML("<h3>Últimos 5 registros (Tail)</h3>" + df.tail().to_html())
    plots_out = Output()
    with plots_out:
        display(HTML("<h4>Gráficos de Series de Tiempo Individuales</h4>"))
        for col in df.columns:
            if 'Lag' not in col and 'Return' not in col and 'Volatility' not in col:
                fig, ax = plt.subplots(figsize=(8, 2)); df[col].plot(ax=ax, title=col); plt.tight_layout(); plt.show()
    return VBox([head_html, tail_html, plots_out])

def create_tab_h1_validation(asset):
    asset_results = {'ARIMA': results[asset]['ARIMA_Enriquecido'], 'Random Forest': results[asset]['Random Forest_Enriquecido'], 'Gradient Boosting': results[asset]['Gradient Boosting_Enriquecido']}
    df_res = pd.DataFrame(asset_results).T
    reduction_vs_arima = (1 - df_res.loc['Random Forest', 'MSE'] / df_res.loc['ARIMA', 'MSE']) * 100 if df_res.loc['ARIMA', 'MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())
    graph_out = Output()
    with graph_out:
        fig, ax = plt.subplots(figsize=(10, 5)); test_dates = test_df.index
        ax.plot(test_dates, test_df[f'{asset}_Close'], label='Precio Real', color='black', linewidth=2)
        ax.plot(test_dates, arima_predictions[asset], label='ARIMA (Tradicional)', color='red', linestyle='--')
        ax.plot(test_dates, models[asset]['Random Forest'].predict(X_test_dict[asset]), label='Random Forest (Supervisado)', color='blue', linestyle='--')
        ax.set_title(f'Comparación Visual de Predicciones para {asset}'); ax.legend(); plt.tight_layout(); plt.show()
    return VBox([HTML("<h3>Validación Hipótesis 1: Superioridad de Modelos Supervisados</h3><p>Ligada al <b>Objetivo Específico 4</b>.</p>"), HTML("<hr><h4>Evidencia Cuantitativa</h4>"), styled_table_html, HTML(f"<div style='margin-top:10px; padding:10px; background-color:#d4edda; border-radius:5px;'><b>Conclusión Cuantitativa:</b> Random Forest reduce el error (MSE) en un <b>{reduction_vs_arima:.2f}%</b> vs. ARIMA para {asset}.</div>"), HTML("<hr><h4>Evidencia Visual</h4>"), graph_out, HTML("<p><b>Interpretación Visual:</b> ARIMA (rojo) no captura la tendencia, mientras Random Forest (azul) se adapta a la dinámica real.</p>"), HTML("<hr><h3 style='color:green;'>Veredicto: Hipótesis 1 Aceptada</h3>")])

def create_tab_objective1(asset):
    res_base = results[asset]['RF_Base_Macro']; res_completo = results[asset]['RF_Con_Tasa']
    df_res = pd.DataFrame([res_base, res_completo], index=['Modelo Base', 'Modelo con Tasa Int.']); styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue'); ax1.set_ylabel(f'Precio {asset}', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Treasury_10Y'], color='red', linestyle='--'); ax2.set_ylabel('Tasa 10Y (%)', color='red'); ax2.tick_params(axis='y', labelcolor='red'); plt.title(f'Contexto: Precio vs. Tasa de Interés'); plt.show()
    results_plot_out = Output()
    with results_plot_out:
        fig, ax = plt.subplots(figsize=(5, 3)); bars = ax.bar(df_res.index, df_res['MSE'], color=['gray', 'skyblue']); ax.set_title(f'Reducción de Error'); ax.bar_label(bars, fmt='{:,.2f}'); plt.show()
    return VBox([HTML("<h4>Validación H2: Impacto de Tasas de Interés</h4>"), context_plot_out, HBox([styled_table_html, results_plot_out])])

def create_tab_objective2(asset):
    res_sin = results[asset]['RF_Con_Tasa']; res_con = results[asset]['RF_Con_Inflacion']
    df_res = pd.DataFrame([res_sin, res_con], index=['Modelo SIN Inflación', 'Modelo CON Inflación'])
    incremento_error = (res_sin['MSE'] / res_con['MSE'] - 1) * 100 if res_con['MSE'] > 0 else 0
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').to_html())
    context_plot_out = Output()
    with context_plot_out:
        fig, ax1 = plt.subplots(figsize=(8, 3)); ax1.plot(df_final.index, df_final[f'{asset}_Close'], color='blue'); ax1.set_ylabel(f'Precio {asset}', color='blue'); ax1.tick_params(axis='y', labelcolor='blue'); ax2 = ax1.twinx(); ax2.plot(df_final.index, df_final['Inflacion_T5YIE'], color='green', linestyle='--'); ax2.set_ylabel('Inflación 5A (%)', color='green'); ax2.tick_params(axis='y', labelcolor='green'); plt.title(f'Contexto: Precio vs. Inflación'); plt.show()
    return VBox([HTML("<h4>Validación H3: Impacto de la Inflación</h4>"), context_plot_out, styled_table_html, HTML(f"<div style='margin-top:10px; padding:10px; background-color:#f8d7da; border-radius:5px;'>Eliminar la inflación <b>aumentó el error en un {incremento_error:.2f}%</b> para {asset}.</div>")])

def create_tab_objective3(asset):
    res_base = results[asset]['RF_Base_Macro']; res_enriquecido = results[asset]['Random Forest_Enriquecido']
    df_res = pd.DataFrame([res_base, res_enriquecido], index=['Modelo SIN Históricos', 'Modelo CON Históricos'])
    r2_color = 'color:red' if df_res.loc['Modelo CON Históricos', 'R2'] < 0 else 'color:green'
    html_resumen = (f"<div style='background-color:#d4edda; padding:10px; border-radius:5px;'><b>¡Resultado Clave para {asset}!</b> El R² pasó de {res_base['R2']:.2f} a <b style='{r2_color}'>{res_enriquecido['R2']:.2f}</b>.</div>") if asset == 'AMZN' else ""
    styled_table_html = HTML(df_res.style.format('{:.4f}').highlight_min(subset='MSE', color='#d4edda').apply(lambda x: ['color:red' if v < 0 else 'color:green' for v in x], subset=['R2']).to_html())
    return VBox([HTML("<h4>Validación H4: Impacto de Datos Históricos</h4>"), HTML(html_resumen), styled_table_html])

def create_tab_interpretability(asset):
    model = models[asset]['Random Forest']; X_test = X_test_dict[asset]; X_train = X_train_dict[asset]
    shap_out = Output()
    with shap_out:
        explainer = shap.TreeExplainer(model); shap_values = explainer(X_test); plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_values, X_test, show=False, plot_size=None); plt.title(f'SHAP Summary Plot para {asset}'); plt.tight_layout(); plt.show()
    pdp_out = Output()
    with pdp_out:
        features_to_plot = ['Treasury_10Y', 'Inflacion_T5YIE', 'VIX', f'{asset}_Close_Lag_1']
        fig, ax = plt.subplots(1, len(features_to_plot), figsize=(15, 3))
        PartialDependenceDisplay.from_estimator(model, X_train, features_to_plot, ax=ax); plt.suptitle(f'Partial Dependence Plots (PDP) para {asset}'); plt.tight_layout(); plt.show()
    return VBox([HTML("<h3>Interpretabilidad del Modelo Random Forest</h3>"), HTML("<h4>Gráfico SHAP</h4>"), shap_out, HTML("<hr><h4>Gráficos PDP</h4>"), pdp_out])

# --- NUEVA FUNCIÓN CORREGIDA para la Pestaña de Reporte ---
def create_final_report_tab(asset):
    # Datos para H1
    h1_results = {'ARIMA': results[asset]['ARIMA_Enriquecido'], 'Random Forest': results[asset]['Random Forest_Enriquecido']}
    h1_df = pd.DataFrame(h1_results).T
    h1_reduction = (1 - h1_df.loc['Random Forest', 'MSE'] / h1_df.loc['ARIMA', 'MSE']) * 100 if h1_df.loc['ARIMA', 'MSE'] > 0 else 0
    h1_table = HTML(h1_df.style.format('{:.2f}').to_html())

    # Datos para H2
    h2_res_base = results[asset]['RF_Base_Macro']; h2_res_completo = results[asset]['RF_Con_Tasa']
    h2_df = pd.DataFrame([h2_res_base, h2_res_completo], index=['Modelo Base', 'Modelo con Tasa Int.'])
    h2_table = HTML(h2_df.style.format('{:.2f}').to_html())

    # Datos para H3
    h3_res_sin = results[asset]['RF_Con_Tasa']; h3_res_con = results[asset]['RF_Con_Inflacion']
    h3_incremento_error = (h3_res_sin['MSE'] / h3_res_con['MSE'] - 1) * 100 if h3_res_con['MSE'] > 0 else 0

    # Datos para H4
    h4_res_base = results[asset]['RF_Base_Macro']; h4_res_enriquecido = results[asset]['Random Forest_Enriquecido']
    h4_df = pd.DataFrame([h4_res_base, h4_res_enriquecido], index=['Modelo SIN Históricos', 'Modelo CON Históricos'])
    h4_table = HTML(h4_df.style.format('{:.2f}').to_html())

    # Construcción del VBox con widgets HTML separados
    report_widgets = [
        HTML(f"<h2>Reporte de Validación de Hipótesis para: {asset}</h2><hr>"),
        HTML("<h3>1. Objetivo 4 y Hipótesis 1: Superioridad de Modelos Supervisados</h3>"),
        HTML('<p><b>Hipótesis 1:</b> Los modelos supervisados ofrecen mayor precisión que los tradicionales. &nbsp; <b style="color:green;">VEREDICTO: ACEPTADA</b></p>'),
        HTML("<h4>Evidencia Cuantitativa:</h4>"),
        h1_table,
        HTML(f"<p><b>Interpretación:</b> El modelo Random Forest reduce el error (MSE) en un <b>{h1_reduction:.2f}%</b> en comparación con ARIMA.</p><hr>"),

        HTML("<h3>2. Objetivo 1 y Hipótesis 2: Impacto de las Tasas de Interés</h3>"),
        HTML('<p><b>Hipótesis 2:</b> Las tasas de interés tienen un impacto significativo e inverso. &nbsp; <b style="color:orange;">VEREDICTO: PARCIALMENTE ACEPTADA</b></p>'),
        HTML("<h4>Evidencia Cuantitativa:</h4>"),
        h2_table,
        HTML("<p><b>Interpretación:</b> El impacto es medible y el modelo aprendió una relación compleja para GOOGL, pero fue ambiguo para AMZN, demostrando que no es universal.</p><hr>"),

        HTML("<h3>3. Objetivo 2 y Hipótesis 3: Impacto de la Inflación</h3>"),
        HTML('<p><b>Hipótesis 3:</b> Los índices de inflación influyen significativamente. &nbsp; <b style="color:green;">VEREDICTO: ACEPTADA</b></p>'),
        HTML(f"<h4>Evidencia Cuantitativa (Estudio de Ablación):</h4><p>Al eliminar la variable de inflación, el error del modelo (MSE) aumentó en un <b>{h3_incremento_error:.2f}%</b>.</p><hr>"),

        HTML("<h3>4. Objetivo 3 y Hipótesis 4: Incidencia de los Datos Históricos</h3>"),
        HTML('<p><b>Hipótesis 4:</b> La incorporación de datos históricos mejora la precisión. &nbsp; <b style="color:green; font-weight:bold;">VEREDICTO: ACEPTADA CON CONTUNDENCIA</b></p>'),
        HTML("<h4>Evidencia Cuantitativa:</h4>"),
        h4_table,
        HTML(f"<p><b>Interpretación:</b> La ingeniería de características transformó un modelo inútil (R² de {h4_res_base['R2']:.2f}) en uno con capacidad predictiva real (R² de {h4_res_enriquecido['R2']:.2f} para AMZN).</p>")
    ]

    return VBox(report_widgets)

def create_tab_simulador():
    # (código sin cambios)
    asset_selector_sim = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Seleccione el Activo:')
    tasa_interes_slider = widgets.FloatSlider(value=4.0, min=2.0, max=6.0, step=0.05, description='Tasa Interés:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    inflacion_slider = widgets.FloatSlider(value=2.5, min=1.5, max=4.0, step=0.05, description='Inflación:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    vix_slider = widgets.FloatSlider(value=15, min=10, max=40, step=1, description='VIX:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    precio_anterior_slider = widgets.FloatSlider(value=df_final['AMZN_Close'].iloc[-1], min=150, max=250, step=0.5, description='Precio Anterior:', style={'description_width': 'initial'}, layout=Layout(width='45%'), continuous_update=True)
    tasa_interes_text, inflacion_text, vix_text, precio_anterior_text = (widgets.FloatText(layout=Layout(width='20%')) for _ in range(4))
    link((tasa_interes_slider, 'value'), (tasa_interes_text, 'value')); link((inflacion_slider, 'value'), (inflacion_text, 'value')); link((vix_slider, 'value'), (vix_text, 'value')); link((precio_anterior_slider, 'value'), (precio_anterior_text, 'value'))
    table_output_container = HTML(); graph_output_container = Output()

    def update_simulador(asset, tasa_interes, inflacion, vix, precio_anterior):
        if any(v is None for v in [tasa_interes, inflacion, vix, precio_anterior]): return
        last_known = df_final.iloc[-1].copy(); input_data = pd.DataFrame([last_known])
        input_data['Treasury_10Y'], input_data['Inflacion_T5YIE'], input_data['VIX'], input_data[f'{asset}_Close_Lag_1'] = tasa_interes, inflacion, vix, precio_anterior
        for lag in range(2, 6): input_data[f'{asset}_Close_Lag_{lag}'] = precio_anterior
        features_enriched = [f for f in df_final.columns if f.startswith(asset) or f in features_base_all]; features_enriched = [f for f in features_enriched if f not in [f'{asset}_Close', f'{asset}_Return_1d']]
        input_data_final = input_data[features_enriched]
        preds = {name: model.predict(input_data_final)[0] for name, model in models[asset].items() if name != 'ARIMA_pred_estatica'}; preds['ARIMA'] = models[asset]['ARIMA_pred_estatica']
        table_output_container.value = create_html_table(preds, precio_anterior)
        graph_output_container.clear_output(wait=True)
        with graph_output_container: create_prediction_graph(df_final[f'{asset}_Close'], preds, asset, precio_anterior)

    def create_html_table(preds, precio_anterior):
        best_model_name = "Random Forest"
        html = f"""<div style="border:1px solid #ccc;padding:15px;border-radius:8px;"><h3 style="margin-top:0;">Tabla de Predicciones</h3><p style="font-size:0.9em;">Precio Anterior: <b>{precio_anterior:,.2f} USD</b></p><table style="width:100%; border-collapse: collapse;"><thead><tr style="background-color:#005a9c;color:white;"><th style="padding:10px;text-align:left;">Modelo</th><th style="padding:10px;text-align:right;">Precio Predicho</th></tr></thead><tbody>"""
        for model_name in ['Random Forest', 'Gradient Boosting', 'Regresión Lineal', 'ARIMA']:
            pred = preds[model_name]; is_best = (model_name == best_model_name); style = 'background-color:#d4edda;color:#155724;font-weight:bold;' if is_best else ''
            tag = ' <span style="font-size:0.8em;background-color:#28a745;color:white;padding:2px 5px;border-radius:3px;">Mejor Modelo</span>' if is_best else ''
            static_tag = ' <span style="font-size:0.8em;background-color:#dc3545;color:white;padding:2px 5px;border-radius:3px;">Estático</span>' if model_name == 'ARIMA' else ''
            html += f"""<tr style="{style}"><td style="padding:8px;border-bottom:1px solid #ddd;">{model_name}{tag}{static_tag}</td><td style="padding:8px;border-bottom:1px solid #ddd;text-align:right;">{pred:,.2f}</td></tr>"""
        return html + "</tbody></table></div>"

    def create_prediction_graph(historic_data_series, preds, asset, precio_anterior):
        plt.style.use('seaborn-v0_8-whitegrid'); fig, ax = plt.subplots(figsize=(9, 5)); historic_data = historic_data_series.tail(100)
        ax.plot(historic_data.index, historic_data.values, label=f'Histórico de {asset}', color='black', alpha=0.7)
        last_date = historic_data.index[-1]; prediction_date = last_date + pd.Timedelta(days=1); colors = {'ARIMA': 'red', 'Regresión Lineal': 'orange', 'Gradient Boosting': 'purple', 'Random Forest': 'blue'}; best_model_name = "Random Forest"
        for model_name, prediction in preds.items():
            is_best = (model_name == best_model_name)
            ax.plot([last_date, prediction_date], [precio_anterior, prediction], linestyle='--', marker='o', color=colors[model_name], label=f'{model_name}: {prediction:,.2f}', linewidth=3.0 if is_best else 1.5, alpha=1.0 if is_best else 0.8)
        ax.set_title(f'Pronósticos para {asset} bajo Escenario Simulado', fontsize=16); ax.set_ylabel('Precio (USD)'); ax.legend(loc='best'); plt.tight_layout(); plt.show()

    def on_asset_change_simulador(change):
        asset = change['new']; last_price = df_final[f'{asset}_Close'].iloc[-1]
        precio_anterior_slider.min = last_price * 0.8; precio_anterior_slider.max = last_price * 1.2
        precio_anterior_slider.value = last_price

    asset_selector_sim.observe(on_asset_change_simulador, names='value')
    controles_sliders = VBox([HBox([tasa_interes_slider, tasa_interes_text]), HBox([inflacion_slider, inflacion_text]), HBox([vix_slider, vix_text]), HBox([precio_anterior_slider, precio_anterior_text])])
    controles_principales = VBox([asset_selector_sim, HTML("<hr><p><b>Defina un escenario:</b></p>"), controles_sliders])
    resultados_ui = HBox([table_output_container, graph_output_container], layout=Layout(align_items='flex-start', justify_content='space-between'))
    widgets.interactive_output(update_simulador, {'asset': asset_selector_sim, 'tasa_interes': tasa_interes_slider, 'inflacion': inflacion_slider, 'vix': vix_slider, 'precio_anterior': precio_anterior_slider})
    on_asset_change_simulador({'new': asset_selector_sim.value})
    return VBox([HTML("<h3>Simulador de Escenarios en Vivo</h3>"), controles_principales, resultados_ui])

# ==============================================================================
# PASO 5: MONTAJE FINAL DEL DASHBOARD
# ==============================================================================
tab_children = [Output() for _ in range(8)]
tabs = widgets.Tab(children=tab_children)
tabs.set_title(0, '0. Exploración'); tabs.set_title(1, '1. Validación H1'); tabs.set_title(2, '2. Obj. 1: Tasas Int.'); tabs.set_title(3, '3. Obj. 2: Inflación'); tabs.set_title(4, '4. Obj. 3: D. Históricos'); tabs.set_title(5, '5. Interpretabilidad'); tabs.set_title(6, '6. REPORTE FINAL'); tabs.set_title(7, '7. DEMO INTERACTIVA')
asset_selector_static = widgets.Dropdown(options=['AMZN', 'GOOGL'], value='AMZN', description='Activo para Análisis:')

def update_all_tabs(change):
    asset = change['new']
    with tab_children[0]: tab_children[0].clear_output(wait=True); display(create_tab_exploration())
    with tab_children[1]: tab_children[1].clear_output(wait=True); display(create_tab_h1_validation(asset))
    with tab_children[2]: tab_children[2].clear_output(wait=True); display(create_tab_objective1(asset))
    with tab_children[3]: tab_children[3].clear_output(wait=True); display(create_tab_objective2(asset))
    with tab_children[4]: tab_children[4].clear_output(wait=True); display(create_tab_objective3(asset))
    with tab_children[5]: tab_children[5].clear_output(wait=True); display(create_tab_interpretability(asset))
    with tab_children[6]: tab_children[6].clear_output(wait=True); display(create_final_report_tab(asset))

asset_selector_static.observe(update_all_tabs, names='value')

with tab_children[7]:
    tab_children[7].clear_output(wait=True)
    display(create_tab_simulador())

display(HTML("<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las pestañas 0-6. La pestaña 7 (DEMO) tiene su propio selector.</p>")); display(asset_selector_static); display(tabs)
update_all_tabs({'new': 'AMZN'})

Instalando librerías necesarias... (puede tardar un momento la primera vez)
✅ Datos cargados correctamente.
✅ Ingeniería de características completada.
✅ Todos los modelos han sido entrenados y evaluados.


HTML(value='<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las…

Dropdown(description='Activo para Análisis:', options=('AMZN', 'GOOGL'), value='AMZN')

Instalando librerías necesarias... (puede tardar un momento la primera vez)
✅ Datos cargados correctamente.
✅ Ingeniería de características completada.
✅ Todos los modelos han sido entrenados y evaluados.


HTML(value='<h1>Dashboard de Tesis: Análisis y Simulación</h1><p>Use el selector para cambiar el activo en las…

Dropdown(description='Activo para Análisis:', options=('AMZN', 'GOOGL'), value='AMZN')